# Battle Royale: pUC/ColE1 copy-number × host-burden simulation

Colab-ready analysis notebook for the Thai SynBio Challenge / Battle Royale copy-number thought experiment.

**Scope and firewall.** Everything here comes from published literature, public competition structure, and declared synthetic scenarios. The 20-lineage Arena is a synthetic scenario: competitor copy numbers, cargo parameters and plasmid architectures are invented for the model and are **not** inferred from participant materials. No participant-submitted strategy, team name, or aggregate derived from presentations appears anywhere in this notebook. Later sections carry short reminders where a specific output could be misread as derived; this paragraph is the general statement and is not repeated.

Core first-order burden model:

\[
r(C)=1-\beta C
\]

with endpoint carrier-plasmid abundance approximated by

\[
Q(T)\propto N_0\,2^{G(C)}\,C\,R(T).
\]

If plasmid loss is parameterized as a **per-division loss probability** \(\ell\),

\[
R(T)\approx (1-\ell)^{G(C)}
\]

in the simple fixed-\(G_0\) approximation.

The notebook treats all numerical optima as **scenario calculations**, not BL21(DE3)-specific predictions and not forecasts of the competition winner.

Primary literature anchor: [@rouches2022tunable]. Its coefficient is stated with its full parameterisation under MODEL STATUS below.

## Model layers

1. **Analytic fixed-growth-opportunity model** for transparent intuition and closed-form checks.
2. **Shared-resource batch model** with plasmid loss and a declared synthetic reference pool.
3. **pUC-specific fixed-PCN extension** that separates DNA amount (PCN × plasmid length) from a generic saturating cargo-expression burden.
4. **Mechanistic extension** with substrate depletion, dynamic copy-number–growth feedback, explicit equal-share inoculation, and optional PCN-dependent loss.
5. **Single-cell PCN heterogeneity** with stochastic copy-number control and partitioning.
6. **Cross-model QA** with fault-sensitive guards rather than tautological checks.

## Scenario conventions

The 20-strain equal-inoculum baseline is the reference scenario throughout. Whenever an Arena-like counterfactual is shown under equal inoculation, each lineage begins at

\[
f_i(0)=1/20=0.05.
\]

Displayed focal strategies are compared in **separate 20-strain counterfactual flasks** against the same 19-member synthetic pUC/ColE1 reference pool, so changing which focal PCNs are drawn cannot silently change their starting densities. Barcode fraction is not forced to start at 0.05: at time zero it is molecule-weighted and therefore already depends on PCN.

The declared “all-pUC” modelling layer is a simplifying assumption, not a claim about submitted constructs. The notebook applies fault-sensitive validation away from the pUC19 reference length, refuses parameter combinations for which the first-order burden law predicts zero/negative growth, harmonizes loss-rate framing, and allows non-20-team synthetic pool-size sensitivity without changing the canonical 20-team baseline.


## MODEL STATUS

What this notebook treats as measured, declared, exploratory, and out of
scope. Read this before any graph.

| | |
| --- | --- |
| **CALIBRATED / MEASURED** | The pUC19 per-copy burden coefficient of Rouches et al. [@rouches2022tunable], used with its parameterisation (below). Literature measurements from [@shao2021singlecell], [@jahn2016copynumber], [@akeno2014reducedgenome], [@wang2006cole1], [@klumpp2011growthrate] and [@sezonov2007lb] are used as reference points, not refitted here. |
| **DECLARED SCENARIO** | 20 equal-inoculum lineages; a 19-member PCN-100 reference competitor pool; *K*/*N*₀, culture duration and starting fractions. These are chosen, not measured. |
| **EXPLORATORY MECHANISMS** | Linear plasmid-length transfer of the burden coefficient; saturating cargo burden; dynamic copy-number feedback; PCN-dependent loss; single-cell regulation noise. Each is a hypothesis generator. |
| **NOT CLAIMED** | The actual distribution of participant designs; a BL21(DE3)-specific winning PCN; the competition winner; an exact mapping from simulated plasmid abundance to barcode read count. |

### The burden coefficient, with its parameterisation

The single calibrated input deserves its full statement, because the bare
percentage is not usable on its own.

Rouches et al. [@rouches2022tunable] fit

$$\mathrm{cost} = a\,b\,x + b$$

where *x* is plasmid copy number, **a = 0.065% ± 0.006%** and
**b = 0.019 min⁻¹ ± 0.003 min⁻¹**, across **830 priming-promoter pUC19
variants** that are *"all with nearly identical origins of replications"*.

Four things follow, and all four matter for how far the number travels:

1. ***a* is dimensionless.** *x* is a copy count and *b* carries min⁻¹, so
   *a* is a fraction, not an absolute rate. "0.065% per copy" is a fraction
   **of *b***.
2. ***b* is never defined in words in the paper.** Its value and units are
   given; what quantity it represents is not stated. No absolute per-copy
   rate is quoted in this notebook, because its baseline cannot be recovered
   from the source.
3. **The authors interpret *a*** as the fraction of cellular growth
   resources one plasmid copy consumes, comparing 0.065% with pUC19's 0.058%
   of the *E. coli* genome length. That interpretation is theirs, and is
   what gives the fraction standalone meaning.
4. **The linear form is hedged by its authors**, who say it only *"largely"*
   agrees with their data. CALIBRATED applies to the coefficient inside that
   form, not to linearity itself.

The paper's abstract gives 0.063%; the Results and Fig. 4E give
0.065% ± 0.006%. This notebook uses the Results value, which is the one
attributed to a specific figure. The fit population is the 830-variant
priming-promoter sub-library, not the full 1,194-variant library.

Not covered by CALIBRATED: that the burden is linear; an absolute per-copy
rate; transfer to other replication origins, other hosts, or cargo-bearing
plasmids. Sections that extend the coefficient along plasmid length or add
cargo are **exploratory**, and say so where they appear.


In [ ]:
# @title Optional Arial registration (portable fallback)
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
_loaded_arial=[]
for path in [Path('/content/Arial.ttf'),Path('/content/Arial_Bold.ttf')]:
    if path.exists() and path.stat().st_size:
        try:
            fm.fontManager.addfont(str(path)); _loaded_arial.append(str(path))
        except Exception as exc:
            warnings.warn(f'Arial unavailable: {exc}',RuntimeWarning)
plt.rcParams['font.family']='Arial' if _loaded_arial else 'DejaVu Sans'
print('Font:',plt.rcParams['font.family'])


In [ ]:
# @title Optional scientific colormaps
try:
    import cmcrameri.cm as cmc
except ImportError:
    cmc=None


In [ ]:
# @title Configuration
# Core numerical / plotting imports
import warnings
warnings.simplefilter("always", RuntimeWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)
from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar
from matplotlib.colors import LinearSegmentedColormap, Normalize

# -----------------------------
# LITERATURE-ANCHORED PARAMETERS
# -----------------------------
# Rouches et al. Fig. 4E fit: a = 0.065% ± 0.006% per plasmid copy.
# Decimal form:
BETA = 0.00065 # @param {type:"number"}
BETA_SD_PUBLISHED = 0.00006

# Approximate range directly explored by the ColE1 library.
VALIDATED_PCN_MAX = 800

# -----------------------------
# DEFAULT SCENARIO PARAMETERS
# -----------------------------
REFERENCE_G0 = 10.0 # @param {type:"number"}
LOW_PCN = 80
HIGH_PCN = 500

# Probability that a carrier lineage loses the plasmid per completed carrier doubling.
# Set to 0 for the baseline copy-number-only analysis.
LOSS_PER_DIVISION = 0.0 # @param {type:"number"}
LOSS_SWEEP = "0.0,0.001,0.01"

# -----------------------------
# ADVANCED / FIXED DEFAULTS
# Edit these directly only when a specific sensitivity analysis needs them.
# -----------------------------
# DYNAMIC COPY-UP PARAMETERS
DYNAMIC_C_EARLY = 80
DYNAMIC_C_TARGET = 500

# log2(copy-number increase) attainable per one unit of reference growth opportunity.
# 1.0 means 80 -> 160 -> 320 -> 640 over ~3 reference-doubling units after activation.
AMP_RATE_LOG2_PER_REF_DOUBLING = 1.0

# SHARED-FLASK PARAMETERS
SHARED_HOURS = 12.0  # illustrative; official duration not specified
ARENA_TEAM_COUNT = 20
EQUAL_START_FRACTION = 1.0 / ARENA_TEAM_COUNT
REFERENCE_POOL_PCNS = np.repeat(100.0, ARENA_TEAM_COUNT - 1)
REFERENCE_POOL_LABEL = "Synthetic pUC/ColE1 reference pool: 19 competitors, each PCN 100"
FOCAL_FRACTION = EQUAL_START_FRACTION
FOCAL_FRACTION_LABEL = "focal fraction 1/20 = 0.05 (illustrative equal-cell inoculum)"
COMPARISON_ANCHOR_PCN = 100.0
REFERENCE_DOUBLING_TIME_H = 0.5
TOTAL_STARTING_CELLS = 1e6 # @param {type:"number"}
CARRYING_CAPACITY = 1e9 # @param {type:"number"}
MATCHED_G0 = np.log2(CARRYING_CAPACITY / TOTAL_STARTING_CELLS)
PLASMID_FREE_GROWTH_MULTIPLIER = 1.0

# pUC/ColE1-specific extension.
# Rouches' fitted per-copy burden used a 2,686-bp pUC19 reference. Length scaling
# below is therefore a sensitivity mechanism, not a re-fitted universal law.
PUC19_BP = 2686.0
PUC_DEFAULT_PLASMID_BP = PUC19_BP
PUC_CARGO_HALF_SAT_PCN = 100.0
PUC_DEFAULT_CARGO_MAX = 0.0  # fractional growth penalty approached at very high PCN
PUC_LOSS_REFERENCE_PCN = 100.0
PUC_LOSS_PCN_EXPONENT = 0.0  # 0 = constant loss; >0 makes low-PCN loss higher

# A fully synthetic 20-member engineered-pUC continuum used only for sensitivity.
SYNTHETIC_20_PCN = np.geomspace(20.0, VALIDATED_PCN_MAX, ARENA_TEAM_COUNT)

# MECHANISTIC RESOURCE + PCN FEEDBACK MODEL
# Comma-separated starting PCNs for the mechanistic shared-flask scenario.
MECHANISTIC_PCN_LIST = "50,100,200,500"

# Monod half-saturation constant for the NORMALIZED substrate S in [0,1].
# This is a scenario parameter, not a fitted LB constant.
MONOD_KS = 0.05

# Strength of positive growth-rate -> copy-number feedback.
# Default is deliberately conservative because no universal BL21/pUC fit exists.
# C_ss ~ C_ref * (mu_ref / mu)^gamma.
# Literature supports growth-rate dependence, but this gamma is phenomenological.
COPY_FEEDBACK_GAMMA = 0.25 # @param {type:"number"}

# Relaxation timescale for copy number to approach its current set point.
COPY_RELAXATION_H = 1.0 # @param {type:"number"}

# Explicit biological/sensitivity bound on growth-state-mediated copy-up.
MAX_COPY_SETPOINT_FOLD = 2.0

# DNA replication may remain possible at lower substrate than bulk population growth.
# Phenomenological rather than a fitted LB constant.
COPY_REPLICATION_KS = 0.005 # @param {type:"number"}

# Maximum absolute PCN allowed in the feedback model.
DYNAMIC_PCN_CAP = 800

# SINGLE-CELL HETEROGENEITY MODEL
SINGLE_CELL_MEAN_PCN = 100
SINGLE_CELL_N = 10000

# Initial coefficient of variation of PCN.
# Illustrative: Jahn et al. motivates heterogeneity but does not define a universal CV.
SINGLE_CELL_INITIAL_CV = 0.30 # @param {type:"number"}

# Multiplicative stochasticity in copy-number regulation per sqrt(hour).
SINGLE_CELL_CONTROL_NOISE = 0.15 # @param {type:"number"}

SINGLE_CELL_DT_H = 0.10
SINGLE_CELL_SEED = 42

# EDITORIAL COMMUNICATION STYLE
EDITORIAL_MODE = True
SHOW_MAJOR_Y_GRID = True

# -----------------------------
# FIGURE SYSTEM — scientific geometry + editorial hierarchy
# -----------------------------
# Implements the data-viz method: form first, colour by the job it does,
# validated palette, recessive chrome, direct labels, light + dark.

FIGURE_THEME = "light"

_T = {
 "light": dict(surface="#fcfcfb", ink="#0b0b0b", ink2="#52514e", muted="#b8b8b8",
               cat=["#2a78d6","#eb6834","#1baf7a","#eda100"],
               seq=["#b7d3f6","#86b6ef","#5598e7","#2a78d6","#256abf","#184f95","#0d366b"]),
 "dark":  dict(surface="#1a1a19", ink="#f0efec", ink2="#c3c2b7", muted="#5a5a57",
               cat=["#3987e5","#d95926","#199e70","#c98500"],
               seq=["#9ec5f4","#6da7ec","#3987e5","#256abf","#1c5cab","#184f95","#104281"]),
}
# Categorical slots validated (light & dark) on the adjacent pairlist:
#   worst CVD dE 9.2 / 9.4, worst normal-vision dE 27.6 / 26.5.
# #1baf7a is below 3:1 on the light surface -> relief rule: always direct-label it.

def theme(k): return _T[FIGURE_THEME][k]

def seq_colors(n, lo=1, hi=6):
    """Interpolated sequential ramp for an ORDERED quantity.
    lo/hi refer to positions along the 7-anchor theme ramp."""
    s = theme("seq")
    cmap = LinearSegmentedColormap.from_list("theme_seq", s)
    denom = max(len(s) - 1, 1)
    positions = np.linspace(lo / denom, hi / denom, n)
    return [cmap(float(np.clip(p, 0, 1))) for p in positions]

def new_fig(w=9, h=5.6, ncols=1):
    fig, axes = plt.subplots(1, ncols, figsize=(w, h), facecolor=theme("surface"))
    return fig, axes

def clean_axes(ax, y_grid=None, x_grid=False):
    y_grid = SHOW_MAJOR_Y_GRID if y_grid is None else y_grid
    ax.grid(False)
    ax.set_facecolor(theme("surface"))
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(theme("muted")); ax.spines[s].set_linewidth(0.8)
    ax.tick_params(colors=theme("ink2"), labelsize=10, length=3, width=0.8)
    if y_grid: ax.grid(axis="y", color=theme("muted"), alpha=0.35, linewidth=0.7)
    if x_grid: ax.grid(axis="x", color=theme("muted"), alpha=0.35, linewidth=0.7)
    ax.set_axisbelow(True)
    ax.xaxis.label.set_color(theme("ink2")); ax.yaxis.label.set_color(theme("ink2"))
    ax.xaxis.set_major_locator(MaxNLocator(6)); ax.yaxis.set_major_locator(MaxNLocator(6))

def fig_header(fig, title, subtitle=None, x=0.02):
    """Title states the finding; subtitle states the method and the caveat."""
    if not EDITORIAL_MODE:
        fig.suptitle(title, fontsize=12)
        return
    fig.suptitle(title, fontsize=15, color=theme("ink"), x=x, ha="left", y=1.0, weight="bold")
    if subtitle:
        fig.text(x, 0.925, subtitle, fontsize=10.2, color=theme("ink2"), ha="left")

def direct_label(ax, x, y, text, color, dx=8, dy=0, ha="left", weight="bold"):
    ax.annotate(text, (x, y), xytext=(dx, dy), textcoords="offset points",
                ha=ha, va="center", fontsize=10.5, color=color, weight=weight)

def note(ax, x, y, text, ha="left", va="center"):
    ax.text(x, y, text, fontsize=9.5, color=theme("ink2"), ha=ha, va=va)

# Apply theme to figures that still use standard Matplotlib constructors.
from matplotlib.ticker import MaxNLocator
plt.rcParams.update({
    "font.family": "Arial" if ("_loaded_arial" in globals() and _loaded_arial) else "sans-serif",
    "figure.facecolor": theme("surface"),
    "axes.facecolor": theme("surface"),
    "savefig.facecolor": theme("surface"),
    "text.color": theme("ink"),
    "axes.labelcolor": theme("ink2"),
    "xtick.color": theme("ink2"),
    "ytick.color": theme("ink2"),
    "axes.titlecolor": theme("ink"),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

# Compact aliases used by existing figure cells.
ACCENT = theme("cat")[0]
MUTED = theme("muted")
DARK = theme("ink")

# Continuous PCN colormap built from the validated sequential theme ramp.
PCN_CMAP = LinearSegmentedColormap.from_list("pcn_seq", theme("seq"))

# Continuous colormaps remain perceptually ordered.
try:
    _cmc_available = cmc is not None
except NameError:
    _cmc_available = False
DIV_CMAP = cmc.vik if _cmc_available else plt.get_cmap("PuOr_r")

# Backward-compatible wrappers; new report figures use fig_header/direct_label.
def editorial_title(title, subtitle=None, ax=None):
    if ax is None:
        ax = plt.gca()
    fig = ax.figure
    fig_header(fig, title, subtitle)
    return ax

def direct_label_last(x, y, label, color=None, ax=None, dx=0.01, dy=0.0):
    if ax is None:
        ax = plt.gca()
    if color is None:
        color = theme("ink")
    xr = np.nanmax(x) - np.nanmin(x)
    ax.text(x[-1] + dx*xr, y[-1] + dy, label, color=color, va="center", fontsize=9, clip_on=False)


# -----------------------------
# OUTPUT
# -----------------------------
SAVE_FIGURES = False
FIG_DIR = Path("battle_royale_figures")
FIG_DIR.mkdir(exist_ok=True)

def savefig(name):
    if SAVE_FIGURES:
        plt.savefig(FIG_DIR / name, dpi=300, bbox_inches="tight")

print("Reference growth opportunity from K/N0:",
      MATCHED_G0,
      "doublings")

def parse_pcn_list(value):
    vals = [int(float(x.strip())) for x in str(value).split(",") if x.strip()]
    if not vals:
        raise ValueError("PCN list is empty.")
    return vals

def parse_float_list(value):
    vals = [float(x.strip()) for x in str(value).split(",") if x.strip()]
    if not vals:
        raise ValueError("Float list is empty.")
    return vals

MECHANISTIC_PCNS = parse_pcn_list(MECHANISTIC_PCN_LIST)
LOSS_SWEEP_VALUES = parse_float_list(LOSS_SWEEP)
print("Mechanistic PCNs:", MECHANISTIC_PCNS)
print("Loss sweep:", LOSS_SWEEP_VALUES)

## Figure system used in this notebook

Quantitative figures use position and length as the primary encodings, with colour redundant wherever possible [@wong2010datafigures; @midway2020visualization; @jambor2025checklist]. The separation of **form** from **colour** follows from that:

- quantitative comparisons are carried primarily by position/length;
- ordered quantities such as generations, loss rate, and copy number use sequential ramps;
- unordered categories use the validated categorical slots;
- signed quantities around a reference use a diverging map;
- series are directly labelled whenever practical, so identity is not colour-only;
- titles state the finding, while subtitles state the method/caveat;
- axes, spines, and gridlines are visually recessive;
- light and dark themes are selected explicitly rather than produced by simple colour inversion.

`FIGURE_THEME` is an advanced configuration setting. The default remains **light** for manuscript/report use; dark mode is available for presentation/export.

## 1. Core analytic model

\(G_0\) is the **reference strain's growth opportunity**, not the number of doublings that every lineage is guaranteed to realize.

For fixed PCN,

\[
G(C)=G_0(1-\beta C).
\]

The simple carrier-only model with per-division segregational loss \(\ell\) is

\[
Q(C)
\propto
2^{G(C)}\,C\,(1-\ell)^{G(C)}.
\]

When \(\ell=0\), the analytic optimum is

\[
C^\ast=\frac{1}{\beta G_0\ln 2}.
\]

The shared-flask model later relaxes the assumption that every lineage experiences an independent growth opportunity.

In [ ]:
# @title Core model functions
def _resolve_beta(beta):
    return BETA if beta is None else beta

def _resolve_loss(loss_per_division):
    return LOSS_PER_DIVISION if loss_per_division is None else loss_per_division

def _validate_loss(loss_per_division):
    loss=float(loss_per_division)
    if not (0 <= loss < 1): raise ValueError('loss_per_division must be in [0, 1).')
    return loss

def _validate_pcn(pcn,beta=None,validated_max=None):
    beta=_resolve_beta(beta)
    if validated_max is None: validated_max=VALIDATED_PCN_MAX
    p=np.asarray(pcn,dtype=float)
    if np.any(p<=0): raise ValueError('PCN must be > 0.')
    zero_growth_pcn=1.0/beta
    if np.any(p>=zero_growth_pcn):
        raise ValueError(f'Linear model predicts zero/negative growth at PCN >= {zero_growth_pcn:.1f}.')
    if np.any(p>validated_max):
        warnings.warn(f'PCN exceeds approximate Rouches library range ({validated_max}).',RuntimeWarning,stacklevel=2)
    return p

def relative_growth(pcn,beta=None):
    beta=_resolve_beta(beta); p=_validate_pcn(pcn,beta=beta); return 1.0-beta*p

def effective_doublings(pcn,g0,beta=None): return np.asarray(g0)*relative_growth(pcn,beta=beta)


def _broadcast_design_parameter(value, k, name, *, positive=False, nonnegative=False):
    arr=np.asarray(value,dtype=float)
    if arr.ndim==0:
        arr=np.repeat(float(arr),k)
    if arr.shape!=(k,) or not np.all(np.isfinite(arr)):
        raise ValueError(f'{name} must be a finite scalar or length-{k} vector.')
    if positive and np.any(arr<=0):
        raise ValueError(f'{name} must be > 0.')
    if nonnegative and np.any(arr<0):
        raise ValueError(f'{name} must be >= 0.')
    return arr

def puc_burden_components(pcn, plasmid_bp=None, cargo_burden_max=None,
                          cargo_half_sat_pcn=None, beta=None):
    """Scenario decomposition for pUC/ColE1-like constructs.

    dna_burden scales the Rouches pUC19 per-copy coefficient by DNA length.
    cargo_burden is a deliberately generic saturating expression penalty.
    The latter is NOT a fitted universal law; it exists to keep expression cost
    separate from DNA-copy cost during sensitivity analysis.
    """
    beta=_resolve_beta(beta)
    p=np.asarray(pcn,dtype=float)
    if np.any(p<=0) or not np.all(np.isfinite(p)):
        raise ValueError('PCN must be finite and > 0.')
    bp=PUC_DEFAULT_PLASMID_BP if plasmid_bp is None else np.asarray(plasmid_bp,dtype=float)
    cargo_max=PUC_DEFAULT_CARGO_MAX if cargo_burden_max is None else np.asarray(cargo_burden_max,dtype=float)
    half=PUC_CARGO_HALF_SAT_PCN if cargo_half_sat_pcn is None else float(cargo_half_sat_pcn)
    if np.any(bp<=0) or np.any(~np.isfinite(bp)): raise ValueError('plasmid_bp must be finite and > 0.')
    if np.any(cargo_max<0) or np.any(cargo_max>=1) or np.any(~np.isfinite(cargo_max)):
        raise ValueError('cargo_burden_max must be finite and in [0,1).')
    if half<=0: raise ValueError('cargo_half_sat_pcn must be > 0.')
    dna=beta*p*(bp/PUC19_BP)
    cargo=cargo_max*p/(half+p)
    # The length/cargo extension is first-order and only meaningful while total
    # predicted burden remains below one. Do not silently floor biologically
    # invalid negative-growth states.
    total=np.asarray(dna+cargo,dtype=float)
    if np.any(total>=1.0):
        p_b,bp_b,dna_b,cargo_b,total_b=np.broadcast_arrays(
            np.asarray(p,dtype=float),np.asarray(bp,dtype=float),
            np.asarray(dna,dtype=float),np.asarray(cargo,dtype=float),total)
        flat=int(np.argmax(total_b.ravel()))
        raise ValueError(
            'Burden model predicts zero or negative growth: '
            f'PCN={float(p_b.ravel()[flat]):.1f}, '
            f'plasmid_bp={float(bp_b.ravel()[flat]):.0f}, '
            f'dna={float(dna_b.ravel()[flat]):.3f} + '
            f'cargo={float(cargo_b.ravel()[flat]):.3f} = '
            f'{float(total_b.ravel()[flat]):.3f}. '
            'The linear first-order law does not extend here.')
    return dna,cargo

def puc_growth_factor(pcn, plasmid_bp=None, cargo_burden_max=None,
                      cargo_half_sat_pcn=None, beta=None):
    dna,cargo=puc_burden_components(
        pcn,plasmid_bp=plasmid_bp,cargo_burden_max=cargo_burden_max,
        cargo_half_sat_pcn=cargo_half_sat_pcn,beta=beta)
    growth=1.0-dna-cargo
    if np.any(growth<=0):
        raise RuntimeError('Internal burden validation failure: non-positive growth escaped validation.')
    return np.clip(growth,1e-12,1.0)

def pcn_dependent_loss_per_division(pcn, base_loss=None, reference_pcn=None,
                                    exponent=None, max_loss=0.95):
    """Phenomenological low-PCN loss sensitivity.

    exponent=0 exactly recovers the constant per-division loss model. Positive
    exponents increase loss below the reference PCN and decrease it above the
    reference. This is a scenario mechanism, not a fitted pUC segregation law.
    """
    base=_validate_loss(_resolve_loss(base_loss))
    ref=PUC_LOSS_REFERENCE_PCN if reference_pcn is None else float(reference_pcn)
    expn=PUC_LOSS_PCN_EXPONENT if exponent is None else float(exponent)
    p=np.asarray(pcn,dtype=float)
    if ref<=0 or expn<0: raise ValueError('reference_pcn > 0 and exponent >= 0 required.')
    if np.any(p<=0): raise ValueError('PCN must be > 0 for PCN-dependent loss.')
    if base==0: return np.zeros_like(p,dtype=float)
    return np.clip(base*np.power(ref/p,expn),0.0,max_loss)

def log2_score(pcn,g0,beta=None,loss_per_division=None,n0=1.0):
    beta=_resolve_beta(beta); loss=_validate_loss(_resolve_loss(loss_per_division))
    p=_validate_pcn(pcn,beta=beta); g=effective_doublings(p,g0,beta=beta)
    return np.log2(n0)+g+np.log2(p)+g*np.log2(1.0-loss)

def score_ratio(pcn_a,pcn_b,g0,beta=None,loss_a=None,loss_b=None):
    beta=_resolve_beta(beta)
    delta=log2_score(pcn_a,g0,beta=beta,loss_per_division=_resolve_loss(loss_a))-log2_score(pcn_b,g0,beta=beta,loss_per_division=_resolve_loss(loss_b))
    return 2.0**delta

def analytic_optimal_pcn(g0,beta=None,loss_per_division=None):
    beta=_resolve_beta(beta); loss=_validate_loss(_resolve_loss(loss_per_division)); g0=np.asarray(g0,dtype=float)
    if np.any(g0<=0): raise ValueError('g0 must be > 0.')
    d=np.log(2.0*(1.0-loss))
    g0, beta = np.broadcast_arrays(g0, np.asarray(beta, dtype=float))
    if np.any(~np.isfinite(beta)) or np.any(beta<=0): raise ValueError('beta must be finite and > 0.')
    if d<=0: return np.full(g0.shape,np.nan,dtype=float)
    return 1.0/(np.asarray(beta)*g0*d)

def monod_activity(substrate,ks):
    if ks<=0: raise ValueError('ks must be > 0.')
    s=np.maximum(np.asarray(substrate,dtype=float),0.0)
    return np.where(s>0,s/(ks+s),0.0)

def copy_setpoint(base_pcn,mu,mu_ref,gamma=None,max_fold=None,pcn_cap=None):
    if gamma is None: gamma=COPY_FEEDBACK_GAMMA
    if max_fold is None: max_fold=MAX_COPY_SETPOINT_FOLD
    if pcn_cap is None: pcn_cap=DYNAMIC_PCN_CAP
    if gamma<0: raise ValueError('gamma must be >= 0.')
    if max_fold<1: raise ValueError('max_fold must be >= 1.')
    base=np.asarray(base_pcn,dtype=float); mu=np.asarray(mu,dtype=float); mu_ref=np.asarray(mu_ref,dtype=float)
    ratio=mu_ref/np.maximum(mu,np.finfo(float).tiny)
    fold=np.minimum(np.power(ratio,gamma),max_fold)
    return np.clip(base*fold,0.0,pcn_cap)

def copy_relaxation_rate(current_pcn,setpoint_pcn,tau_h,activity):
    if tau_h<=0: raise ValueError('tau_h must be > 0.')
    return (np.asarray(setpoint_pcn)-np.asarray(current_pcn))/tau_h*activity

def copy_relaxation_step(current_pcn,setpoint_pcn,dt_h,tau_h,activity):
    if tau_h<=0 or dt_h<0: raise ValueError('tau_h > 0 and dt_h >= 0 required.')
    frac=1.0-np.exp(-np.asarray(activity)*dt_h/tau_h)
    return np.asarray(current_pcn)+frac*(np.asarray(setpoint_pcn)-np.asarray(current_pcn))

def expect_raises(fn,exc_type=Exception):
    try: fn()
    except exc_type: return True
    return False

assert expect_raises(lambda:_validate_pcn(0),ValueError)
assert expect_raises(lambda:_validate_pcn(1.0/BETA),ValueError)
_validation_grid=np.arange(1,VALIDATED_PCN_MAX+1)
for _loss in [0.0,0.001,0.01]:
    _numeric=_validation_grid[np.argmax(log2_score(_validation_grid,REFERENCE_G0,loss_per_division=_loss))]
    _analytic=float(analytic_optimal_pcn(REFERENCE_G0,loss_per_division=_loss))
    assert abs(_numeric-_analytic)<=1.0,(_loss,_numeric,_analytic)
for g0 in [4,6,8,10,12,15,20]: print(f'G0={g0:>2}: analytic optimum ~ {float(analytic_optimal_pcn(g0)):.1f} copies/cell')

In [ ]:
# @title Compounded population cost of copy number
pcn_grid = np.arange(1, VALIDATED_PCN_MAX + 1)
growth = relative_growth(pcn_grid)

growth_table = pd.DataFrame({
    "PCN": [50, 80, 100, 200, 300, 500, 700],
    "Relative growth": relative_growth([50, 80, 100, 200, 300, 500, 700]),
})
growth_table["Growth penalty (%)"] = 100 * (1 - growth_table["Relative growth"])
display(growth_table.round(3))

fig, ax = new_fig(w=9.2, h=5.8)

for g0 in [5, 10, 15]:
    rel = 2 ** (-BETA * pcn_grid * g0)
    focus = np.isclose(g0, REFERENCE_G0)
    color = ACCENT if focus else MUTED
    lw = 2.8 if focus else 1.5

    ax.plot(pcn_grid, rel, color=color, linewidth=lw)
    direct_label_last(
        pcn_grid,
        rel,
        f"{g0} doublings",
        color=color if focus else theme("ink2"),
        ax=ax,
        dx=0.008,
    )

ax.set_xlabel("Plasmid copy number per cell")
ax.set_ylabel("Carrier cells relative to zero-copy reference")
clean_axes(ax)
fig.subplots_adjust(top=0.82, right=0.88)

# Align the left edge of the title/subtitle with the horizontal position
# of the y-axis title (rather than the canvas edge or the y-axis spine).
fig.canvas.draw()
_ylabel_box = ax.yaxis.label.get_window_extent(
    renderer=fig.canvas.get_renderer()
).transformed(fig.transFigure.inverted())
_header_x = 0.5 * (_ylabel_box.x0 + _ylabel_box.x1)

fig_header(
    fig,
    "A modest per-copy cost compounds into a large population deficit",
    "First-order Rouches burden relationship; curves show different reference growth opportunities.",
    x=_header_x,
)

savefig("01_compounded_population_cost_vs_pcn.png")
plt.show()

In [ ]:
# @title Endpoint score across PCN
g0_values = [4, 6, 8, 10, 12, 15, 20]
_colors = seq_colors(len(g0_values), lo=0, hi=6)

fig, ax = new_fig(w=9.2, h=5.8)

_peak_rows = [18, 34]  # stagger labels so nearby optima do not collide
for j, (g0, color) in enumerate(zip(g0_values, _colors)):
    lq = log2_score(pcn_grid, g0, loss_per_division=LOSS_PER_DIVISION)
    relative_q = 2 ** (lq - np.max(lq))
    is_focus = np.isclose(g0, REFERENCE_G0)

    ax.plot(
        pcn_grid,
        relative_q,
        color=color,
        linewidth=2.8 if is_focus else 1.7,
        alpha=1.0 if is_focus else 0.90,
    )

    peak_idx = int(np.argmax(relative_q))
    ax.annotate(
        rf'$G_0$={g0}',
        (pcn_grid[peak_idx], relative_q[peak_idx]),
        xytext=(0, _peak_rows[j % 2]),
        textcoords='offset points',
        ha='center', va='bottom',
        fontsize=9.2,
        color=color,
        weight='bold' if is_focus else 'normal',
        arrowprops=dict(arrowstyle='-', color=color, linewidth=0.7, alpha=0.7),
    )

ax.set_xlabel('Plasmid copy number per cell')
ax.set_ylabel('Endpoint score / best score in scenario')
ax.set_ylim(0, 1.10)
clean_axes(ax)
fig_header(
    fig,
    'More growth opportunity shifts the optimum toward lower copy number',
    r'First-order ColE1 burden model; ordered $G_0$ values use a sequential ramp and are labelled at their optima.',
)
fig.subplots_adjust(top=0.82)
savefig('02_normalized_score_vs_pcn.png')
plt.show()

In [ ]:
# @title Optimal PCN vs available growth
# Start at G0 = 3 so the analytic optimum is largely inside the measured ~1–800-copy range.
g0_grid = np.linspace(3, 25, 300)
c_opt = analytic_optimal_pcn(g0_grid, loss_per_division=LOSS_PER_DIVISION)

plt.figure(figsize=(8, 5))
plt.plot(g0_grid, c_opt)
plt.axhline(VALIDATED_PCN_MAX, linestyle="--", linewidth=1)
plt.xlabel("Reference growth opportunity, G0 (doublings)")
plt.ylabel("Predicted optimal PCN")
plt.title("Optimal copy number shifts downward as growth opportunity increases")
savefig("03_optimal_pcn_vs_generations.png")
plt.show()

opt_table = pd.DataFrame({
    "G0 (reference doublings)": g0_values,
    "Analytic optimal PCN": [analytic_optimal_pcn(g, loss_per_division=LOSS_PER_DIVISION) for g in g0_values],
})
opt_table["Within ~1–800-copy library range"] = (
    opt_table["Analytic optimal PCN"] <= VALIDATED_PCN_MAX
)
display(opt_table.round(1))

In [ ]:
# @title Low-copy vs high-copy break-even
g0_pair = np.linspace(0.01, 25, 300)
ratio = score_ratio(
    LOW_PCN,
    HIGH_PCN,
    g0_pair,
    loss_a=LOSS_PER_DIVISION,
    loss_b=LOSS_PER_DIVISION,
)
log2_ratio = np.log2(ratio)

plt.figure(figsize=(8, 5))
plt.plot(g0_pair, log2_ratio)
plt.axhline(0, linewidth=1)
plt.xlabel("Reference growth opportunity, G0 (doublings)")
plt.ylabel(f"log2(score {LOW_PCN}-copy / {HIGH_PCN}-copy)")
plt.title(f"When does a {LOW_PCN}-copy carrier overtake a {HIGH_PCN}-copy carrier?")
savefig("04_pairwise_break_even.png")
plt.show()

if LOSS_PER_DIVISION == 0:
    delta_r = relative_growth(LOW_PCN) - relative_growth(HIGH_PCN)
    g0_break_even = np.log2(HIGH_PCN / LOW_PCN) / delta_r
    print(f"Break-even G0 ≈ {g0_break_even:.2f} reference doublings")
else:
    print("With nonzero segregational loss, use the plotted numerical break-even.")

print(f"r({LOW_PCN})  = {relative_growth(LOW_PCN):.4f}")
print(f"r({HIGH_PCN}) = {relative_growth(HIGH_PCN):.4f}")

In [ ]:
# @title Candidate vs high-copy heatmap
candidate_pcns = np.arange(20, 301, 5)
g0_heat = np.linspace(2, 20, 181)

heat = np.zeros((len(g0_heat), len(candidate_pcns)))
for i, g0 in enumerate(g0_heat):
    heat[i, :] = [
        np.log2(score_ratio(c, HIGH_PCN, g0, loss_a=LOSS_PER_DIVISION, loss_b=LOSS_PER_DIVISION))
        for c in candidate_pcns
    ]

X, Y = np.meshgrid(candidate_pcns, g0_heat)

plt.figure(figsize=(10, 6))
vmax = np.nanmax(np.abs(heat))
mesh = plt.pcolormesh(
    X, Y, heat,
    shading="auto",
    cmap=DIV_CMAP,
    vmin=-vmax,
    vmax=vmax,
)
plt.colorbar(mesh, label=f"log2(score candidate / score {HIGH_PCN}-copy)")
plt.contour(X, Y, heat, levels=[0], linewidths=1)
plt.xlabel("Candidate PCN")
plt.ylabel("Reference growth opportunity, G0")
plt.title(f"Moderate-copy strategy versus {HIGH_PCN}-copy strategy")
savefig("05_candidate_vs_high_copy_heatmap.png")
plt.show()

## 2. Dynamic copy-up model

The previous notebook effectively awarded the target late PCN even when there was no time to reach it.

Here, copy number is explicitly ramped after activation. In reference-growth coordinates \(g\),

\[
C(g)=
\min\left[
C_{\rm target},
C_{\rm early}\,2^{a(g-g_{\rm switch})}
\right]
\]

after the switch, where \(a\) is the user-adjustable amplification rate in log2 copy-number units per unit reference growth opportunity.

The host-growth contribution is integrated over the resulting \(C(g)\) trajectory. The endpoint score uses the **actually reached** endpoint PCN.

In [ ]:
# @title Dynamic copy-up simulation
def dynamic_copy_profile(
    g_axis,
    g0,
    c_early,
    c_target,
    switch_fraction,
    amp_rate_log2_per_ref_doubling=AMP_RATE_LOG2_PER_REF_DOUBLING,
):
    if not (0 <= switch_fraction <= 1):
        raise ValueError("switch_fraction must be in [0, 1].")
    if amp_rate_log2_per_ref_doubling < 0:
        raise ValueError("Amplification rate must be >= 0.")

    _validate_pcn([c_early, c_target])
    g_switch = switch_fraction * g0
    after = np.maximum(g_axis - g_switch, 0.0)

    c = c_early * 2.0 ** (amp_rate_log2_per_ref_doubling * after)
    c = np.minimum(c, c_target)
    c[g_axis <= g_switch] = c_early
    return c

def dynamic_log2_score(
    c_early,
    c_target,
    g0,
    switch_fraction,
    beta=BETA,
    loss_per_division=None,
    amp_rate_log2_per_ref_doubling=AMP_RATE_LOG2_PER_REF_DOUBLING,
    n_steps=4001,
):
    loss_per_division = _validate_loss(_resolve_loss(loss_per_division))
    g_axis = np.linspace(0, g0, n_steps)
    c_profile = dynamic_copy_profile(
        g_axis,
        g0,
        c_early,
        c_target,
        switch_fraction,
        amp_rate_log2_per_ref_doubling,
    )

    # Integrate actual host doublings across reference-growth opportunity.
    r_profile = relative_growth(c_profile, beta=beta)
    host_doublings = np.trapezoid(r_profile, g_axis)

    # Per-division plasmid loss compounds over completed carrier doublings.
    log2_retention = host_doublings * np.log2(1.0 - loss_per_division)

    c_end = c_profile[-1]
    score = host_doublings + np.log2(c_end) + log2_retention

    return {
        "log2_score": score,
        "host_doublings": host_doublings,
        "endpoint_pcn": c_end,
        "g_axis": g_axis,
        "pcn_profile": c_profile,
    }

switch_fractions = np.linspace(0, 1, 201)
dynamic_results = [
    dynamic_log2_score(
        DYNAMIC_C_EARLY,
        DYNAMIC_C_TARGET,
        REFERENCE_G0,
        f,
        loss_per_division=LOSS_PER_DIVISION,
    )
    for f in switch_fractions
]

dynamic_scores = np.array([x["log2_score"] for x in dynamic_results])
dynamic_end_pcns = np.array([x["endpoint_pcn"] for x in dynamic_results])

fixed_early = log2_score(
    DYNAMIC_C_EARLY,
    REFERENCE_G0,
    loss_per_division=LOSS_PER_DIVISION,
)

# Critical validation: switching at the very end must reduce exactly to fixed-early.
assert np.isclose(dynamic_scores[-1], fixed_early, atol=1e-3), (
    dynamic_scores[-1], fixed_early
)
assert np.isclose(dynamic_end_pcns[-1], DYNAMIC_C_EARLY)

plt.figure(figsize=(8, 5))
plt.plot(switch_fractions, dynamic_scores - fixed_early)
plt.axhline(0, linewidth=1)
plt.xlabel("Fraction of reference growth opportunity before copy-up begins")
plt.ylabel(f"log2(score / fixed {DYNAMIC_C_EARLY}-copy)")
plt.title("Dynamic copy-up with an explicit amplification ramp")
savefig("06_dynamic_copy_up_score.png")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(switch_fractions, dynamic_end_pcns)
plt.xlabel("Fraction of reference growth opportunity before copy-up begins")
plt.ylabel("Actually reached endpoint PCN")
plt.title("Late activation may not have enough time to reach target PCN")
savefig("07_dynamic_copy_up_endpoint_pcn.png")
plt.show()

## 3. Shared-flask model with plasmid loss

For each design \(i\), the shared-resource model tracks:

- \(P_i\): plasmid-bearing carrier cells
- \(F_i\): plasmid-free descendants

Carrier cells grow at their PCN-dependent rate. A per-division plasmid-loss probability is converted to a continuous state-transition hazard. Plasmid-free descendants grow at the reference rate (scaled by the configurable multiplier) and compete for the same carrying capacity.

For Arena-like illustrations, **20 lineages are initialized at equal cell abundance** unless a sensitivity analysis explicitly says otherwise. Thus each starts at 0.05 of the cells. A focal PCN curve is generated in its own counterfactual flask against 19 unchanged synthetic pUC/ColE1 competitors, rather than putting every plotted candidate into one flask.

This section remains deliberately simple: fixed PCN, logistic resource limitation, no toxins, cross-feeding, stationary-phase death, or spatial structure. The later pUC-specific and mechanistic layers relax some of those assumptions.


In [ ]:
# @title Shared-flask simulator
def simulate_shared_batch(pcns,hours=None,reference_doubling_time_h=None,total_n0=None,carrying_capacity=None,beta=None,loss_per_division=None,plasmid_free_growth_multiplier=None,n_points=500,initial_fractions=None):
    hours=SHARED_HOURS if hours is None else hours; reference_doubling_time_h=REFERENCE_DOUBLING_TIME_H if reference_doubling_time_h is None else reference_doubling_time_h
    total_n0=TOTAL_STARTING_CELLS if total_n0 is None else total_n0; carrying_capacity=CARRYING_CAPACITY if carrying_capacity is None else carrying_capacity
    plasmid_free_growth_multiplier=PLASMID_FREE_GROWTH_MULTIPLIER if plasmid_free_growth_multiplier is None else plasmid_free_growth_multiplier
    beta=_resolve_beta(beta); loss_per_division=_validate_loss(_resolve_loss(loss_per_division)); pcns=np.asarray(pcns,dtype=float); _validate_pcn(pcns,beta=beta)
    if carrying_capacity<=total_n0: raise ValueError('carrying_capacity must exceed total_n0.')
    if pcns.ndim != 1 or len(pcns)==0 or not np.all(np.isfinite(pcns)): raise ValueError("pcns must be a nonempty finite vector.")
    k=len(pcns); mu0=np.log(2)/reference_doubling_time_h; mu_carrier=mu0*relative_growth(pcns,beta=beta); mu_free=mu0*plasmid_free_growth_multiplier
    loss_hazard_full=np.zeros(k) if loss_per_division==0 else -np.log(1-loss_per_division)*mu_carrier/np.log(2)
    fractions = np.full(k,1/k) if initial_fractions is None else np.asarray(initial_fractions,float)
    if fractions.shape != (k,) or not np.all(np.isfinite(fractions)) or np.any(fractions<=0) or not np.isclose(fractions.sum(),1.0,rtol=0,atol=1e-10):
        raise ValueError('initial_fractions must be finite, positive, match pcns and sum to one.')
    p0=total_n0*fractions; f0=np.zeros(k); y0=np.concatenate([p0,f0])
    def ode(t,y):
        P=y[:k]; F=y[k:]; total=np.sum(P)+np.sum(F); resource=max(0.0,1.0-total/carrying_capacity); transfer=loss_hazard_full*resource*P
        return np.concatenate([mu_carrier*resource*P-transfer,mu_free*resource*F+transfer])
    t_eval=np.linspace(0,hours,n_points); sol=solve_ivp(ode,(0,hours),y0,t_eval=t_eval,rtol=1e-8,atol=1e-10)
    if not sol.success: raise RuntimeError(sol.message)
    if not np.all(np.isfinite(sol.y)): raise RuntimeError('Non-finite values returned by ODE solver.')
    P=sol.y[:k]; F=sol.y[k:]; total_by_design=P+F; total_all=np.sum(total_by_design,axis=0); resource_t=np.clip(1-total_all/carrying_capacity,0,1)
    barcode_signal=P*pcns[:,None]; denom=np.maximum(np.sum(total_by_design,axis=0,keepdims=True),1e-300); carrier_cell_fraction=P/denom; design_cell_fraction=total_by_design/denom
    barcode_fraction=barcode_signal/np.maximum(np.sum(barcode_signal,axis=0,keepdims=True),1e-300)
    gross=np.array([np.trapezoid(mu_carrier[i]*resource_t/np.log(2),sol.t) for i in range(k)]); net=np.log2(np.maximum(P[:,-1],1e-300)/P[:,0])
    return {'time_h':sol.t,'carriers':P,'plasmid_free':F,'design_total':total_by_design,'carrier_cell_fraction':carrier_cell_fraction,'design_cell_fraction':design_cell_fraction,'barcode_fraction':barcode_fraction,'pcns':pcns,'mu_carrier':mu_carrier,'resource':resource_t,'gross_carrier_doublings':gross,'net_carrier_log2_change':net}

def evaluate_focal_metrics(candidate_pcn,pool=None,focal_fraction=None,**kwargs):
    """Evaluate one focal candidate against a declared fixed pool.

    Returns separate biological and measurement-proxy readouts so absolute plasmid
    yield is never silently conflated with within-pool barcode fraction.
    """
    pool=REFERENCE_POOL_PCNS if pool is None else np.asarray(pool,float)
    focal_fraction=FOCAL_FRACTION if focal_fraction is None else float(focal_fraction)
    if pool.ndim != 1 or len(pool)==0 or not np.all(np.isfinite(pool)):
        raise ValueError('pool must be a nonempty finite vector.')
    if not 0<focal_fraction<1:
        raise ValueError('0 < focal_fraction < 1 required.')
    fractions=np.r_[focal_fraction,np.repeat((1-focal_fraction)/len(pool),len(pool))]
    result=simulate_shared_batch(np.r_[float(candidate_pcn),pool],initial_fractions=fractions,**kwargs)
    carrier=float(result['carriers'][0,-1])
    absolute_plasmids=carrier*float(candidate_pcn)
    barcode_fraction=float(result['barcode_fraction'][0,-1])
    lineage_fraction=float(result['design_cell_fraction'][0,-1])
    total_cells=float(np.sum(result['design_total'][:,-1]))
    k_used=CARRYING_CAPACITY if kwargs.get('carrying_capacity',None) is None else float(kwargs['carrying_capacity'])
    return {
        'carrier_cells':carrier,
        'absolute_plasmids':absolute_plasmids,
        'barcode_fraction':barcode_fraction,
        'lineage_fraction':lineage_fraction,
        'total_cells':total_cells,
        'occupancy':total_cells/k_used,
    }

def evaluate_focal(candidate_pcn,pool=None,focal_fraction=None,**kwargs):
    """Backward-compatible two-column view: carrier cells, absolute plasmids."""
    m=evaluate_focal_metrics(candidate_pcn,pool=pool,focal_fraction=focal_fraction,**kwargs)
    return m['carrier_cells'],m['absolute_plasmids']

def focal_grid(candidates,**kwargs):
    return np.array([evaluate_focal(c,**kwargs) for c in candidates])

def focal_metric_grid(candidates,metric='absolute_plasmids',**kwargs):
    allowed={'carrier_cells','absolute_plasmids','barcode_fraction','lineage_fraction'}
    if metric not in allowed: raise ValueError(f'metric must be one of {sorted(allowed)}')
    return np.array([evaluate_focal_metrics(c,**kwargs)[metric] for c in candidates],dtype=float)

def bounded_focal_optimum(metric='absolute_plasmids',lower=10.0,upper=None,**kwargs):
    """Resolution-independent bounded optimum for one declared focal readout."""
    upper=float(VALIDATED_PCN_MAX if upper is None else upper); lower=float(lower)
    if not lower<upper: raise ValueError('lower must be < upper.')
    def value(c): return evaluate_focal_metrics(c,**kwargs)[metric]
    opt=minimize_scalar(lambda c:-value(c),bounds=(lower,upper),method='bounded')
    if not opt.success: raise RuntimeError(opt.message)
    options=[lower,float(opt.x),upper]
    best=max(options,key=value)
    return float(best),float(value(best))


# --- pUC-specific fixed-PCN extension -------------------------------------------------
def simulate_puc_fixed_batch(pcns, plasmid_bp=None, cargo_burden_max=None,
                             cargo_half_sat_pcn=None, hours=None,
                             reference_doubling_time_h=None, total_n0=None,
                             carrying_capacity=None, beta=None,
                             loss_per_division=None, loss_reference_pcn=None,
                             loss_pcn_exponent=None, plasmid_free_growth_multiplier=None,
                             n_points=500, initial_fractions=None):
    """Fixed-PCN shared flask with DNA-length and generic cargo burdens separated."""
    hours=SHARED_HOURS if hours is None else float(hours)
    reference_doubling_time_h=REFERENCE_DOUBLING_TIME_H if reference_doubling_time_h is None else float(reference_doubling_time_h)
    total_n0=TOTAL_STARTING_CELLS if total_n0 is None else float(total_n0)
    carrying_capacity=CARRYING_CAPACITY if carrying_capacity is None else float(carrying_capacity)
    plasmid_free_growth_multiplier=PLASMID_FREE_GROWTH_MULTIPLIER if plasmid_free_growth_multiplier is None else float(plasmid_free_growth_multiplier)
    beta=_resolve_beta(beta)
    base_loss=_validate_loss(_resolve_loss(loss_per_division))
    pcns=np.asarray(pcns,dtype=float)
    _validate_pcn(pcns,beta=beta)
    if pcns.ndim!=1 or len(pcns)==0 or not np.all(np.isfinite(pcns)):
        raise ValueError('pcns must be a nonempty finite vector.')
    if carrying_capacity<=total_n0: raise ValueError('carrying_capacity must exceed total_n0.')
    k=len(pcns)
    bp=_broadcast_design_parameter(PUC_DEFAULT_PLASMID_BP if plasmid_bp is None else plasmid_bp,k,'plasmid_bp',positive=True)
    cargo=_broadcast_design_parameter(PUC_DEFAULT_CARGO_MAX if cargo_burden_max is None else cargo_burden_max,k,'cargo_burden_max',nonnegative=True)
    if np.any(cargo>=1): raise ValueError('cargo_burden_max must be < 1.')
    fractions=np.full(k,1/k) if initial_fractions is None else np.asarray(initial_fractions,float)
    if fractions.shape!=(k,) or not np.all(np.isfinite(fractions)) or np.any(fractions<=0) or not np.isclose(fractions.sum(),1.0,rtol=0,atol=1e-10):
        raise ValueError('initial_fractions must be finite, positive, match pcns and sum to one.')
    dna_burden,cargo_burden=puc_burden_components(pcns,bp,cargo,cargo_half_sat_pcn,beta)
    growth_factor=puc_growth_factor(pcns,bp,cargo,cargo_half_sat_pcn,beta)
    mu0=np.log(2)/reference_doubling_time_h
    mu_carrier=mu0*growth_factor
    mu_free=mu0*plasmid_free_growth_multiplier
    loss_div=pcn_dependent_loss_per_division(pcns,base_loss,loss_reference_pcn,loss_pcn_exponent)
    loss_hazard=np.where(loss_div>0,-np.log(1-loss_div)*mu_carrier/np.log(2),0.0)
    p0=total_n0*fractions; f0=np.zeros(k); y0=np.concatenate([p0,f0])
    def ode(t,y):
        P=np.maximum(y[:k],0); F=np.maximum(y[k:],0)
        total=np.sum(P)+np.sum(F)
        resource=max(0.0,1.0-total/carrying_capacity)
        transfer=loss_hazard*resource*P
        return np.concatenate([mu_carrier*resource*P-transfer,mu_free*resource*F+transfer])
    sol=solve_ivp(ode,(0,hours),y0,t_eval=np.linspace(0,hours,n_points),rtol=1e-8,atol=1e-10)
    if not sol.success: raise RuntimeError(sol.message)
    if not np.all(np.isfinite(sol.y)): raise RuntimeError('Non-finite values returned by ODE solver.')
    P=np.maximum(sol.y[:k],0); F=np.maximum(sol.y[k:],0); total=P+F
    total_all=np.sum(total,axis=0)
    barcode_signal=P*pcns[:,None]
    return {
        'time_h':sol.t,'pcns':pcns,'plasmid_bp':bp,'cargo_burden_max':cargo,
        'dna_burden':dna_burden,'cargo_burden':cargo_burden,'growth_factor':growth_factor,
        'loss_per_division_effective':loss_div,'carriers':P,'plasmid_free':F,
        'design_total':total,
        'design_cell_fraction':total/np.maximum(total_all[None,:],1e-300),
        'barcode_fraction':barcode_signal/np.maximum(np.sum(barcode_signal,axis=0,keepdims=True),1e-300),
    }

def simulate_puc_focal_fixed(candidate_pcn, candidate_bp=None, candidate_cargo=None,
                             pool_pcns=None, pool_bp=None, pool_cargo=None,
                             focal_fraction=None, **kwargs):
    pool=REFERENCE_POOL_PCNS if pool_pcns is None else np.asarray(pool_pcns,float)
    if pool.ndim!=1 or len(pool)<1 or not np.all(np.isfinite(pool)):
        raise ValueError('pool_pcns must be a nonempty finite vector.')
    # Canonical Arena baseline is 19 competitors + 1 focal strain. For explicit
    # pool-size sensitivity, allow other sizes but default to equal inoculation
    # for the actual number of lineages rather than preserving an inconsistent 5%.
    if len(pool)!=ARENA_TEAM_COUNT-1:
        warnings.warn(
            f'Pool has {len(pool)} competitors; declared Arena baseline is {ARENA_TEAM_COUNT-1}. '
            'Using a synthetic pool-size scenario.',RuntimeWarning,stacklevel=2)
    ff=(1.0/(len(pool)+1)) if focal_fraction is None else float(focal_fraction)
    if not 0<ff<1:
        raise ValueError('0 < focal_fraction < 1 required.')
    pcns=np.r_[float(candidate_pcn),pool]
    bp=np.r_[PUC_DEFAULT_PLASMID_BP if candidate_bp is None else float(candidate_bp),
             np.repeat(PUC_DEFAULT_PLASMID_BP if pool_bp is None else float(pool_bp),len(pool))]
    cargo=np.r_[PUC_DEFAULT_CARGO_MAX if candidate_cargo is None else float(candidate_cargo),
                np.repeat(PUC_DEFAULT_CARGO_MAX if pool_cargo is None else float(pool_cargo),len(pool))]
    fractions=np.r_[ff,np.repeat((1-ff)/len(pool),len(pool))]
    return simulate_puc_fixed_batch(pcns,plasmid_bp=bp,cargo_burden_max=cargo,
                                    initial_fractions=fractions,**kwargs)

def evaluate_puc_focal_fixed(candidate_pcn, candidate_bp=None, candidate_cargo=None,
                             metric='absolute_plasmids', **kwargs):
    sim=simulate_puc_focal_fixed(candidate_pcn,candidate_bp=candidate_bp,
                                 candidate_cargo=candidate_cargo,**kwargs)
    carrier=float(sim['carriers'][0,-1]); absolute=carrier*float(candidate_pcn)
    vals={
        'carrier_cells':carrier,
        'absolute_plasmids':absolute,
        'barcode_fraction':float(sim['barcode_fraction'][0,-1]),
        'lineage_fraction':float(sim['design_cell_fraction'][0,-1]),
    }
    if metric not in vals: raise ValueError(f'unknown metric {metric}')
    return vals[metric]

def _max_valid_puc_pcn(plasmid_bp=None,cargo_burden_max=None,cargo_half_sat_pcn=None,beta=None,
                       search_upper=None):
    """Largest PCN inside the positive-growth domain of the first-order pUC law."""
    bp=PUC_DEFAULT_PLASMID_BP if plasmid_bp is None else float(plasmid_bp)
    cargo=PUC_DEFAULT_CARGO_MAX if cargo_burden_max is None else float(cargo_burden_max)
    half=PUC_CARGO_HALF_SAT_PCN if cargo_half_sat_pcn is None else float(cargo_half_sat_pcn)
    beta=float(_resolve_beta(beta))
    upper=float(VALIDATED_PCN_MAX if search_upper is None else search_upper)
    def burden(c): return beta*c*(bp/PUC19_BP)+cargo*c/(half+c)
    if burden(upper)<1.0:
        return upper
    # monotone burden law: bisection is deterministic and avoids invalid ODE calls
    lo,hi=0.0,upper
    for _ in range(80):
        mid=(lo+hi)/2
        if burden(mid)<1.0: lo=mid
        else: hi=mid
    return np.nextafter(lo,0.0)

def bounded_puc_focal_optimum(candidate_bp=None,candidate_cargo=None,metric='absolute_plasmids',
                              lower=10.0,upper=None,**kwargs):
    requested_upper=float(VALIDATED_PCN_MAX if upper is None else upper)
    safe_upper=_max_valid_puc_pcn(
        candidate_bp,candidate_cargo,kwargs.get('cargo_half_sat_pcn',None),kwargs.get('beta',None),
        search_upper=requested_upper)
    lower=float(lower)
    if not lower<safe_upper:
        raise ValueError(f'No positive-growth search interval: lower={lower:g}, valid upper={safe_upper:g}.')
    def value(c): return evaluate_puc_focal_fixed(c,candidate_bp,candidate_cargo,metric=metric,**kwargs)
    opt=minimize_scalar(lambda c:-value(c),bounds=(lower,safe_upper),method='bounded')
    if not opt.success: raise RuntimeError(opt.message)
    options=[lower,float(opt.x),safe_upper]
    best=max(options,key=value)
    return float(best),float(value(best))


In [ ]:
# @title Shared-flask trajectories — equal-share 20-strain counterfactuals
example_pcns = [50, 80, 100, 150, 200, 300, 500, 700]
_equal20 = np.repeat(EQUAL_START_FRACTION, ARENA_TEAM_COUNT)
_shared_focal_runs = {
    int(c): simulate_puc_focal_fixed(c, n_points=500)
    for c in example_pcns
}

_pcn_norm = Normalize(vmin=min(example_pcns), vmax=max(example_pcns))
_pcn_colors = {int(c): PCN_CMAP(_pcn_norm(c)) for c in example_pcns}

# A. Focal lineage fraction. Every curve starts at exactly 0.05 cells/cells.
fig, ax = new_fig(w=9.5, h=6.0)
for c in example_pcns:
    sim_c=_shared_focal_runs[int(c)]
    y=sim_c['design_cell_fraction'][0]
    color=_pcn_colors[int(c)]
    ax.plot(sim_c['time_h'],y,color=color,linewidth=2.0,alpha=0.95)
    direct_label_last(sim_c['time_h'],y,f'{int(c)}',color=color,ax=ax,dx=0.012)
ax.axhline(EQUAL_START_FRACTION,color=theme('muted'),linewidth=1.0,linestyle='--')
ax.set_xlabel('Time (h)')
ax.set_ylabel('Focal lineage fraction')
clean_axes(ax)
fig_header(
    fig,
    'Every focal lineage starts at 5%; growth then separates the trajectories',
    'Each curve is a separate 20-strain flask: one focal pUC/ColE1 design + 19 synthetic PCN-100 competitors.',
    x=ax.get_position().x0,
)
fig.subplots_adjust(top=0.82,right=0.88)
savefig('08_shared_batch_lineage_fractions.png')
plt.show()

# B. Barcode fraction. Equal CELL fractions do not imply equal MOLECULE fractions at t=0.
fig, ax = new_fig(w=9.5, h=6.0)
_barcode_end={int(c):float(_shared_focal_runs[int(c)]['barcode_fraction'][0,-1]) for c in example_pcns}
_barcode_order=sorted(example_pcns,key=lambda c:_barcode_end[int(c)])
_barcode_dy={int(c):float(dy) for c,dy in zip(_barcode_order,np.linspace(-14,14,len(_barcode_order)))}
for c in example_pcns:
    sim_c=_shared_focal_runs[int(c)]
    y=sim_c['barcode_fraction'][0]
    color=_pcn_colors[int(c)]
    ax.plot(sim_c['time_h'],y,color=color,linewidth=2.0,alpha=0.95)
    direct_label(ax,sim_c['time_h'][-1],y[-1],f'{int(c)}',color,dx=8,dy=_barcode_dy[int(c)],weight='normal')
ax.set_xlabel('Time (h)')
ax.set_ylabel('Focal plasmid/barcode fraction')
clean_axes(ax)
fig_header(
    fig,
    'Equal 5% cell inocula can begin with unequal barcode shares',
    'At t=0 the molecule-weighted barcode fraction already depends on focal PCN; all other competitors remain PCN 100.',
    x=ax.get_position().x0,
)
fig.subplots_adjust(top=0.82,right=0.88)
savefig('09_shared_batch_barcode_fractions.png')
plt.show()

_start_lineage={c:float(_shared_focal_runs[c]['design_cell_fraction'][0,0]) for c in _shared_focal_runs}
assert all(np.isclose(v,EQUAL_START_FRACTION) for v in _start_lineage.values())
print('Verified focal cell starting fraction for every displayed counterfactual:',EQUAL_START_FRACTION)


## 4. Mechanistic pUC extension: substrate depletion + dynamic \(C(t)\) + separated burdens

This layer is intentionally more biological than the baseline logistic model, but several parameters remain phenomenological.

### Equal-share Arena architecture

A focal mechanistic trajectory is evaluated against the same 19-member synthetic pUC/ColE1 reference pool. Under the equal-inoculum baseline, every lineage starts at \(1/20=0.05\). This removes the old coupling between “how many curves were plotted” and starting density.

### Resource dynamics

A normalized utilizable substrate \(S(t)\) is consumed by population growth:

\[
\mu_{\rm env}(S)=\mu_{\max}\frac{S}{K_S+S}.
\]

The normalization is chosen so that approximately one unit of substrate supports the biomass increase from the starting population toward the configured carrying-capacity scale. This is **not** a chemically explicit LB model.

### DNA amount and cargo expression are separated

For a pUC/ColE1-like lineage with PCN \(C_i\), plasmid length \(L_i\), and a generic cargo burden amplitude \(E_i\), the notebook uses

\[
b_{\rm DNA}=\beta C_i\frac{L_i}{2686},
\]

and a deliberately generic saturating cargo term

\[
b_{\rm cargo}=E_i\frac{C_i}{K_E+C_i}.
\]

The instantaneous carrier growth multiplier is clipped from

\[
1-b_{\rm DNA}-b_{\rm cargo}.
\]

Rouches et al. anchor the pUC19 per-copy coefficient; Akeno et al. support total exogenous DNA (PCN × length) as an informative burden axis. **The exact linear length transfer and the saturating cargo law are sensitivity assumptions, not fitted BL21(DE3) laws.**

### Dynamic copy-number feedback

The copy-number set point can respond to growth state:

\[
C_{ss,i}=\min[C_{\max}, C_{ref,i}\min((\mu_{ref,i}/\mu_i)^\gamma,F_{\max})].
\]

Copy-number relaxation uses a second Monod-like activity term for plasmid replication. Klumpp supports growth-rate-dependent ColE1 copy-number control, but the compact \(\gamma,\tau_C\) law remains phenomenological.

### Optional PCN-dependent loss

The baseline keeps a constant per-division loss probability. A sensitivity switch can instead use

\[
\ell(C)=\ell_{100}\left(\frac{100}{C}\right)^h,
\]

with clipping at high loss. Setting \(h=0\) exactly recovers the constant-loss model. The exponent is a **scenario mechanism**, not a fitted segregation law.

References used for the extensions are listed in the final literature section.


In [ ]:
# @title Mechanistic substrate + dynamic-PCN simulator
def validate_mechanistic_raw_state(raw_C,raw_S,pcn_cap,tol=1e-6):
    raw_C=np.asarray(raw_C,dtype=float); raw_S=np.asarray(raw_S,dtype=float)
    if not np.all(np.isfinite(raw_C)) or not np.all(np.isfinite(raw_S)): raise RuntimeError('Mechanistic solver produced non-finite states.')
    if np.min(raw_S)<-tol: raise RuntimeError(f'Substrate became materially negative: min={np.min(raw_S):.3e}')
    if np.min(raw_C)<-tol: raise RuntimeError(f'PCN became materially negative: min={np.min(raw_C):.3e}')
    if np.max(raw_C)>pcn_cap+tol: raise RuntimeError(f'PCN exceeded configured cap: max={np.max(raw_C):.3f}')
    return True

def simulate_mechanistic_batch(pcns,hours=None,reference_doubling_time_h=None,total_n0=None,
                               carrying_capacity=None,beta=None,monod_ks=None,copy_replication_ks=None,
                               feedback_gamma=None,copy_relaxation_h=None,max_copy_setpoint_fold=None,
                               pcn_cap=None,loss_per_division=None,loss_reference_pcn=None,
                               loss_pcn_exponent=None,plasmid_free_growth_multiplier=None,
                               plasmid_bp=None,cargo_burden_max=None,cargo_half_sat_pcn=None,
                               initial_fractions=None,n_points=700):
    hours=SHARED_HOURS if hours is None else float(hours)
    reference_doubling_time_h=REFERENCE_DOUBLING_TIME_H if reference_doubling_time_h is None else float(reference_doubling_time_h)
    total_n0=TOTAL_STARTING_CELLS if total_n0 is None else float(total_n0)
    carrying_capacity=CARRYING_CAPACITY if carrying_capacity is None else float(carrying_capacity)
    monod_ks=MONOD_KS if monod_ks is None else float(monod_ks)
    copy_replication_ks=COPY_REPLICATION_KS if copy_replication_ks is None else float(copy_replication_ks)
    feedback_gamma=COPY_FEEDBACK_GAMMA if feedback_gamma is None else float(feedback_gamma)
    copy_relaxation_h=COPY_RELAXATION_H if copy_relaxation_h is None else float(copy_relaxation_h)
    max_copy_setpoint_fold=MAX_COPY_SETPOINT_FOLD if max_copy_setpoint_fold is None else float(max_copy_setpoint_fold)
    pcn_cap=DYNAMIC_PCN_CAP if pcn_cap is None else float(pcn_cap)
    plasmid_free_growth_multiplier=PLASMID_FREE_GROWTH_MULTIPLIER if plasmid_free_growth_multiplier is None else float(plasmid_free_growth_multiplier)
    beta=_resolve_beta(beta); base_loss=_validate_loss(_resolve_loss(loss_per_division))
    pcns_ref=np.asarray(pcns,dtype=float); _validate_pcn(pcns_ref,beta=beta)
    if pcns_ref.ndim!=1 or len(pcns_ref)==0: raise ValueError('pcns must be a nonempty vector.')
    if np.any(pcns_ref>pcn_cap): raise ValueError('Starting PCN cannot exceed pcn_cap.')
    if monod_ks<=0 or copy_replication_ks<=0: raise ValueError('Monod constants must be > 0.')
    if copy_relaxation_h<=0: raise ValueError('copy_relaxation_h must be > 0.')
    if carrying_capacity<=total_n0: raise ValueError('carrying_capacity must exceed total_n0.')
    k=len(pcns_ref)
    bp=_broadcast_design_parameter(PUC_DEFAULT_PLASMID_BP if plasmid_bp is None else plasmid_bp,k,'plasmid_bp',positive=True)
    cargo=_broadcast_design_parameter(PUC_DEFAULT_CARGO_MAX if cargo_burden_max is None else cargo_burden_max,k,'cargo_burden_max',nonnegative=True)
    if np.any(cargo>=1): raise ValueError('cargo_burden_max must be < 1.')
    fractions=np.full(k,1/k) if initial_fractions is None else np.asarray(initial_fractions,float)
    if fractions.shape!=(k,) or not np.all(np.isfinite(fractions)) or np.any(fractions<=0) or not np.isclose(fractions.sum(),1.0,rtol=0,atol=1e-10):
        raise ValueError('initial_fractions must be finite, positive, match pcns and sum to one.')

    mu_max=np.log(2)/reference_doubling_time_h
    s0=1.0
    growth_activity0=float(monod_activity(s0,monod_ks))
    mu_env0=mu_max*growth_activity0
    mu_ref=mu_env0*puc_growth_factor(pcns_ref,bp,cargo,cargo_half_sat_pcn,beta)
    p0=total_n0*fractions; f0=np.zeros(k); c0=pcns_ref.copy()
    y0=np.concatenate([p0,f0,c0,[s0]])
    biomass_capacity=carrying_capacity-total_n0

    def ode(t,y):
        P=np.maximum(y[:k],0); F=np.maximum(y[k:2*k],0); C=np.maximum(y[2*k:3*k],1e-9); S=float(y[-1])
        ga=float(monod_activity(S,monod_ks)); ca=float(monod_activity(S,copy_replication_ks)); mu_env=mu_max*ga
        gf=puc_growth_factor(C,bp,cargo,cargo_half_sat_pcn,beta)
        mu_carrier=mu_env*gf
        mu_free=mu_env*plasmid_free_growth_multiplier
        loss_div=pcn_dependent_loss_per_division(C,base_loss,loss_reference_pcn,loss_pcn_exponent)
        transfer=np.zeros(k) if base_loss==0 else (-np.log(1-loss_div))*mu_carrier/np.log(2)*P
        c_ss=copy_setpoint(pcns_ref,mu_carrier,mu_ref,gamma=feedback_gamma,max_fold=max_copy_setpoint_fold,pcn_cap=pcn_cap)
        dC=copy_relaxation_rate(C,c_ss,copy_relaxation_h,ca)
        dP=mu_carrier*P-transfer
        dF=mu_free*F+transfer
        gross=np.sum(mu_carrier*P)+mu_free*np.sum(F)
        dS=-gross/biomass_capacity if S>0 else 0.0
        return np.concatenate([dP,dF,dC,[dS]])

    sol=solve_ivp(ode,(0,hours),y0,t_eval=np.linspace(0,hours,n_points),rtol=1e-7,atol=1e-9)
    if not sol.success: raise RuntimeError(sol.message)
    raw_C=sol.y[2*k:3*k]; raw_S=sol.y[-1]
    validate_mechanistic_raw_state(raw_C,raw_S,pcn_cap)
    P=np.maximum(sol.y[:k],0); F=np.maximum(sol.y[k:2*k],0); C=np.clip(raw_C,0,pcn_cap); S=np.clip(raw_S,0,1)
    ga=monod_activity(S,monod_ks); ca=monod_activity(S,copy_replication_ks); mu_env=mu_max*ga
    mu_carrier=np.array([mu_env*puc_growth_factor(C[j],bp[j],cargo[j],cargo_half_sat_pcn,beta) for j in range(k)])
    barcode_signal=P*C
    barcode_fraction=barcode_signal/np.maximum(np.sum(barcode_signal,axis=0,keepdims=True),1e-300)
    total=P+F
    lineage_fraction=total/np.maximum(np.sum(total,axis=0,keepdims=True),1e-300)
    rel_change=np.max(np.abs(C-pcns_ref[:,None])/pcns_ref[:,None],axis=1)
    return {
        'time_h':sol.t,'pcns_ref':pcns_ref,'plasmid_bp':bp,'cargo_burden_max':cargo,
        'initial_fractions':fractions,'carriers':P,'plasmid_free':F,'pcn':C,
        'substrate':S,'growth_activity':ga,'copy_activity':ca,'mu_env':mu_env,
        'mu_carrier':mu_carrier,'barcode_fraction':barcode_fraction,
        'lineage_fraction':lineage_fraction,'relative_pcn_change':rel_change,
    }

def simulate_mechanistic_focal(candidate_pcn,candidate_bp=None,candidate_cargo=None,
                               pool_pcns=None,pool_bp=None,pool_cargo=None,
                               focal_fraction=None,**kwargs):
    pool=REFERENCE_POOL_PCNS if pool_pcns is None else np.asarray(pool_pcns,float)
    ff=FOCAL_FRACTION if focal_fraction is None else float(focal_fraction)
    if len(pool)!=ARENA_TEAM_COUNT-1:
        raise ValueError(f'Expected {ARENA_TEAM_COUNT-1} reference competitors.')
    pcns=np.r_[float(candidate_pcn),pool]
    bp=np.r_[PUC_DEFAULT_PLASMID_BP if candidate_bp is None else float(candidate_bp),
             np.repeat(PUC_DEFAULT_PLASMID_BP if pool_bp is None else float(pool_bp),len(pool))]
    cargo=np.r_[PUC_DEFAULT_CARGO_MAX if candidate_cargo is None else float(candidate_cargo),
                np.repeat(PUC_DEFAULT_CARGO_MAX if pool_cargo is None else float(pool_cargo),len(pool))]
    fractions=np.r_[ff,np.repeat((1-ff)/len(pool),len(pool))]
    return simulate_mechanistic_batch(pcns,plasmid_bp=bp,cargo_burden_max=cargo,
                                      initial_fractions=fractions,**kwargs)

assert expect_raises(lambda:validate_mechanistic_raw_state(np.array([[100.,100.]]),np.array([1.,-0.1]),DYNAMIC_PCN_CAP),RuntimeError)
assert expect_raises(lambda:validate_mechanistic_raw_state(np.array([[100.,DYNAMIC_PCN_CAP+10.]]),np.array([1.,0.5]),DYNAMIC_PCN_CAP),RuntimeError)

# Matched counterfactuals: each focal starts at 0.05 against the same 19 × PCN100 pool.
mech_runs={int(c):simulate_mechanistic_focal(c) for c in MECHANISTIC_PCNS}
mech=mech_runs.get(100,simulate_mechanistic_focal(100))
print('Reference 20×PCN100 endpoint substrate fraction:',round(float(mech['substrate'][-1]),4))
print('Focal endpoint PCNs:',{c:round(float(run['pcn'][0,-1]),1) for c,run in mech_runs.items()})
print('Focal maximum relative PCN changes:',{c:round(float(run['relative_pcn_change'][0]),3) for c,run in mech_runs.items()})
assert all(np.isclose(run['initial_fractions'][0],EQUAL_START_FRACTION) for run in mech_runs.values())


In [ ]:
# @title Mechanistic model: substrate depletion in the equal-share reference arena
plt.figure(figsize=(8,5)); ax=plt.gca()
ax.plot(mech['time_h'],mech['substrate'],color=ACCENT,linewidth=2.8)
ax.set_xlabel('Time (h)'); ax.set_ylabel('Remaining normalized growth substrate')
editorial_title(
    'The growth window closes as the 20-strain reference arena consumes substrate',
    'Reference case: 20 equal 5% pUC/ColE1 lineages, all starting at PCN 100.',ax=ax)
clean_axes(ax); savefig('10_mechanistic_substrate_depletion.png'); plt.show()


In [ ]:
# @title Mechanistic model: focal dynamic PCN in matched 20-strain arenas
plt.figure(figsize=(9,6)); ax=plt.gca()
_lo,_hi=float(min(MECHANISTIC_PCNS)),float(max(MECHANISTIC_PCNS))
for c0 in MECHANISTIC_PCNS:
    run=mech_runs[int(c0)]
    color=PCN_CMAP(0.15+0.80*(float(c0)-_lo)/max(_hi-_lo,1e-9))
    y=run['pcn'][0]/float(c0)
    ax.plot(run['time_h'],y,color=color,linewidth=2.2)
    direct_label_last(run['time_h'],y,f'{int(c0)}',color=color,ax=ax,dx=0.012)
ax.axhline(1.0,color=theme('ink2'),linewidth=1.0)
ax.set_xlabel('Time (h)'); ax.set_ylabel('Focal mean PCN / starting PCN')
editorial_title(
    'Growth-state feedback is compared without changing focal starting abundance',
    'Every curve is a separate 20-strain counterfactual with focal cell fraction 0.05 and 19 PCN-100 competitors.',ax=ax)
clean_axes(ax); plt.subplots_adjust(right=0.84); savefig('11_mechanistic_dynamic_pcn.png'); plt.show()


### Sensitivity of growth-state copy-up

The growth-state copy-up depends jointly on two **unanchored scenario parameters**: the substrate sensitivity of plasmid replication (`COPY_REPLICATION_KS`) and the copy-number relaxation time (`COPY_RELAXATION_H`). These are evaluated in the same equal-share 20-strain reference arena rather than in a one-strain culture.

The table below uses a 100-copy focal lineage against nineteen 100-copy reference competitors. All twenty begin at 0.05 cell fraction.


In [ ]:
# @title Dynamic-PCN parameter sensitivity in a 20-strain reference arena
_sensitivity_ks=[0.001,0.005,0.02,0.05]
_sensitivity_tau=[0.25,1.0]
_sensitivity_rows=[]
for _tau in _sensitivity_tau:
    for _ks in _sensitivity_ks:
        _sim=simulate_mechanistic_focal(100,copy_replication_ks=_ks,copy_relaxation_h=_tau,n_points=400)
        _end=float(_sim['pcn'][0,-1])
        _sensitivity_rows.append({'COPY_RELAXATION_H':_tau,'COPY_REPLICATION_KS':_ks,'endpoint_PCN':_end,'endpoint_change_%':100*(_end/100-1)})
pcn_feedback_sensitivity=pd.DataFrame(_sensitivity_rows)
display(pcn_feedback_sensitivity.pivot(index='COPY_RELAXATION_H',columns='COPY_REPLICATION_KS',values='endpoint_change_%').round(1))
print('Values are endpoint focal PCN change (%) from a 100-copy start; focal cell fraction starts at 0.05.')


In [ ]:
# @title Mechanistic model: focal barcode share in matched 20-strain arenas
plt.figure(figsize=(9,6)); ax=plt.gca()
rows=[]
for c0 in MECHANISTIC_PCNS:
    run=mech_runs[int(c0)]
    color=PCN_CMAP((float(c0)-min(MECHANISTIC_PCNS))/max(max(MECHANISTIC_PCNS)-min(MECHANISTIC_PCNS),1e-9))
    y=run['barcode_fraction'][0]
    ax.plot(run['time_h'],y,color=color,linewidth=2.2)
    direct_label_last(run['time_h'],y,f'{int(c0)}',color=color,ax=ax,dx=0.012)
    rows.append({
        'starting_PCN':int(c0),
        'starting_cell_fraction':float(run['initial_fractions'][0]),
        'starting_barcode_fraction':float(run['barcode_fraction'][0,0]),
        'endpoint_mean_PCN':float(run['pcn'][0,-1]),
        'endpoint_lineage_fraction':float(run['lineage_fraction'][0,-1]),
        'endpoint_barcode_fraction':float(run['barcode_fraction'][0,-1]),
    })
ax.set_xlabel('Time (h)'); ax.set_ylabel('Focal plasmid/barcode fraction')
editorial_title(
    'Copy number changes both the starting molecule share and subsequent competitive trajectory',
    'Equal 5% cell inocula; each focal PCN is evaluated against the same 19 × PCN100 pUC/ColE1 pool.',ax=ax)
clean_axes(ax); plt.subplots_adjust(right=0.84); savefig('12_mechanistic_barcode_fractions.png'); plt.show()
display(pd.DataFrame(rows).sort_values('starting_PCN').reset_index(drop=True).round(4))


## 5. Single-cell PCN heterogeneity

Population-average PCN can hide very broad cell-to-cell distributions. Shao et al. [@shao2021singlecell] directly counted several common backbones at single-cell resolution and reported a pUC mean near 61 copies with a standard deviation near 60 copies; their pUC distribution therefore had a CV close to 1, with rare plasmid-free cells and a long high-copy tail. Jahn et al. [@jahn2016copynumber] independently showed large subpopulation differences in expression-plasmid PCN by sorting and ddPCR.

The model below is deliberately a **particle approximation**, not a molecular simulation of RNAI/RNAII chemistry. Each virtual cell carries an integer PCN and an inherited copy-control baseline. Over short time steps:

1. cells experience the substrate trajectory from the equal-share 20-strain reference arena;
2. copy-number control relaxes using the same feedback law as the deterministic model;
3. cells divide with a PCN-dependent probability set by their instantaneous growth rate;
4. plasmids are repartitioned stochastically only when a division occurs;
5. the enlarged particle pool is downsampled to a fixed computational population.

The default CV remains conservative at 0.30, and 0.30, 0.60 and 1.00 are swept explicitly later. These are sensitivity values rather than a fitted BL21(DE3)-pUC distribution.

Sources for this section: [@shao2021singlecell], [@jahn2016copynumber].


In [ ]:
# @title Single-cell PCN particle simulation
def _lognormal_integer_sample(mean,cv,n,rng):
    if mean<=0: return np.zeros(n,dtype=int)
    if cv<0: raise ValueError('CV must be >= 0.')
    if cv==0: return np.repeat(int(round(mean)),n)
    sigma2=np.log(1+cv**2); sigma=np.sqrt(sigma2); mu=np.log(mean)-0.5*sigma2
    return np.maximum(np.rint(rng.lognormal(mu,sigma,n)).astype(int),0)

def simulate_single_cell_pcn(mean_pcn=None,n_cells=None,initial_cv=None,control_noise=None,dt_h=None,seed=None,beta=None,feedback_gamma=None,copy_relaxation_h=None,max_copy_setpoint_fold=None,pcn_cap=None,substrate_time=None,substrate_values=None):
    mean_pcn=SINGLE_CELL_MEAN_PCN if mean_pcn is None else mean_pcn; n_cells=SINGLE_CELL_N if n_cells is None else n_cells; initial_cv=SINGLE_CELL_INITIAL_CV if initial_cv is None else initial_cv; control_noise=SINGLE_CELL_CONTROL_NOISE if control_noise is None else control_noise
    dt_h=SINGLE_CELL_DT_H if dt_h is None else dt_h; seed=SINGLE_CELL_SEED if seed is None else seed; beta=_resolve_beta(beta); feedback_gamma=COPY_FEEDBACK_GAMMA if feedback_gamma is None else feedback_gamma; copy_relaxation_h=COPY_RELAXATION_H if copy_relaxation_h is None else copy_relaxation_h; max_copy_setpoint_fold=MAX_COPY_SETPOINT_FOLD if max_copy_setpoint_fold is None else max_copy_setpoint_fold; pcn_cap=DYNAMIC_PCN_CAP if pcn_cap is None else pcn_cap
    rng=np.random.default_rng(seed)
    if substrate_time is None or substrate_values is None:
        substrate_time=np.linspace(0,SHARED_HOURS,500); substrate_values=np.exp(-substrate_time/max(SHARED_HOURS/3,1e-9))
    substrate_time=np.asarray(substrate_time,float); substrate_values=np.asarray(substrate_values,float); t_grid=np.arange(substrate_time.min(),substrate_time.max()+0.5*dt_h,dt_h)
    base_C=_lognormal_integer_sample(mean_pcn,initial_cv,n_cells,rng); C=base_C.copy(); initial_C=C.copy(); mu_max=np.log(2)/REFERENCE_DOUBLING_TIME_H; s0=float(np.interp(t_grid[0],substrate_time,substrate_values)); mu_env0=mu_max*float(monod_activity(s0,MONOD_KS)); mu_ref_base=mu_env0*np.clip(1-beta*base_C,1e-12,1.0); records=[]
    for step,t in enumerate(t_grid):
        S=float(np.clip(np.interp(t,substrate_time,substrate_values),0,1)); ga=float(monod_activity(S,MONOD_KS)); ca=float(monod_activity(S,COPY_REPLICATION_KS)); mu_env=mu_max*ga; mu=mu_env*np.clip(1-beta*C,0,1)
        records.append({'time_h':t,'mean':float(np.mean(C)),'median':float(np.median(C)),'q10':float(np.quantile(C,0.10)),'q90':float(np.quantile(C,0.90)),'cv':float(np.std(C)/max(np.mean(C),1e-12)),'zero_fraction':float(np.mean(C==0)),'mean_growth_h':float(np.mean(mu))})
        if step==len(t_grid)-1: break
        C_ss=copy_setpoint(base_C,mu,mu_ref_base,gamma=feedback_gamma,max_fold=max_copy_setpoint_fold,pcn_cap=pcn_cap); C_ss[C==0]=0; C_cont=copy_relaxation_step(C,C_ss,dt_h,copy_relaxation_h,ca)
        sigma=control_noise*np.sqrt(dt_h)
        if sigma>0: C_cont*=np.exp(rng.normal(-0.5*sigma**2,sigma,size=len(C_cont)))
        C_cont=np.clip(C_cont,0,pcn_cap); p_div=1-np.exp(-np.maximum(mu,0)*dt_h); did=rng.random(len(C))<p_div; non=np.where(~did)[0]; div=np.where(did)[0]
        outC=[]; outB=[]; outM=[]
        if len(non): outC.append(np.rint(C_cont[non]).astype(int)); outB.append(base_C[non]); outM.append(mu_ref_base[non])
        if len(div):
            total=rng.poisson(np.maximum(2*C_cont[div],0)); d1=rng.binomial(total,0.5); d2=total-d1; outC.extend([d1,d2]); outB.extend([base_C[div],base_C[div]]); outM.extend([mu_ref_base[div],mu_ref_base[div]])
        C_pool=np.concatenate(outC).astype(int); B_pool=np.concatenate(outB).astype(int); M_pool=np.concatenate(outM).astype(float); B_pool[C_pool==0]=0; M_pool[C_pool==0]=0
        if len(C_pool)>n_cells:
            keep=rng.choice(len(C_pool),size=n_cells,replace=False); C=C_pool[keep]; base_C=B_pool[keep]; mu_ref_base=M_pool[keep]
        else: C=C_pool; base_C=B_pool; mu_ref_base=M_pool
        C=np.clip(C,0,pcn_cap).astype(int)
    return {'summary':pd.DataFrame(records),'initial_pcn':initial_C,'final_pcn':C.copy(),'final_base_pcn':base_C.copy(),'time_h':t_grid}

single=simulate_single_cell_pcn(substrate_time=mech['time_h'],substrate_values=mech['substrate']); initial_mean=float(np.mean(single['initial_pcn'])); final_mean=float(np.mean(single['final_pcn'])); final_median=float(np.median(single['final_pcn'])); initial_cv_obs=float(np.std(single['initial_pcn'])/max(initial_mean,1e-12)); final_cv_obs=float(np.std(single['final_pcn'])/max(final_mean,1e-12))
print('Initial mean PCN:',round(initial_mean,2)); print('Final mean PCN:',round(final_mean,2)); print('Final median PCN:',round(final_median,2)); print('Initial CV:',round(initial_cv_obs,3)); print('Final CV:',round(final_cv_obs,3)); print('Final plasmid-free fraction:',round(float(np.mean(single['final_pcn']==0)),4))
if np.sum(single['final_pcn'])>0:
    sf=np.sort(single['final_pcn']); nt=max(1,int(np.ceil(0.10*len(sf)))); print('Share of plasmid molecules carried by top 10% PCN cells:',round(float(sf[-nt:].sum()/sf.sum()),3))

In [ ]:
# @title Single-cell PCN: distribution through time
sdf = single["summary"]

plt.figure(figsize=(9, 6))
ax = plt.gca()

ax.fill_between(
    sdf["time_h"],
    sdf["q10"],
    sdf["q90"],
    color=ACCENT,
    alpha=0.14,
    linewidth=0,
    label="10th–90th percentile",
)
ax.plot(
    sdf["time_h"],
    sdf["mean"],
    color=ACCENT,
    linewidth=2.7,
    label="Mean",
)
ax.plot(
    sdf["time_h"],
    sdf["median"],
    color=DARK,
    linewidth=1.8,
    linestyle="--",
    label="Median",
)

ax.set_xlabel("Time (h)")
ax.set_ylabel("Single-cell PCN")
editorial_title(
    "A single mean can hide a broad copy-number distribution",
    "Ribbon shows the 10th–90th percentile of the virtual-cell population.",
    ax=ax,
)
clean_axes(ax)
ax.legend()
savefig("13_single_cell_pcn_distribution_over_time.png")
plt.show()

In [ ]:
# @title Single-cell PCN: initial vs endpoint distribution
initial = single["initial_pcn"]
final = single["final_pcn"]

upper = max(
    10,
    int(np.quantile(np.concatenate([initial, final]), 0.995))
)
bins = np.linspace(0, upper, 45)

plt.figure(figsize=(9, 6))
ax = plt.gca()

ax.hist(
    initial,
    bins=bins,
    density=True,
    histtype="step",
    linewidth=1.7,
    color=MUTED,
    label="Initial",
)
ax.hist(
    final,
    bins=bins,
    density=True,
    histtype="stepfilled",
    alpha=0.28,
    color=ACCENT,
    label="Endpoint",
)
ax.axvline(np.mean(final), color=ACCENT, linewidth=2, label="Endpoint mean")
ax.axvline(np.median(final), color=DARK, linewidth=1.5, linestyle="--", label="Endpoint median")

ax.set_xlabel("PCN per virtual cell")
ax.set_ylabel("Density")
editorial_title(
    "Growth selection and partition noise reshape the within-team PCN distribution",
    "The particle model is qualitative until its variance parameters are calibrated experimentally.",
    ax=ax,
)
clean_axes(ax, y_grid=False)
ax.legend()
savefig("14_single_cell_pcn_endpoint_distribution.png")
plt.show()

### Cross-model validation

The deterministic and particle models call the same copy-control functions. Validation covers **two regimes separately**:

1. a low-noise homogeneous configuration, where the particle mean should closely reproduce the deterministic trajectory (5% tolerance, chosen to catch divergences that a 20% tolerance missed); and
2. the **actual plotted stochastic configuration**, where initial PCN dispersion and control noise create selection against high-PCN cells and are therefore expected to pull the particle mean modestly below the mean-field trajectory. A 10% envelope is used for that biologically different regime.

The validation also requires **material**, not merely directional, responses of endpoint CV to both starting heterogeneity and control noise, and checks timestep robustness.

In [ ]:
# @title Cross-model and stochastic validation
def observed_cv(arr):
    arr = np.asarray(arr, float)
    return float(np.std(arr) / max(np.mean(arr), 1e-12))

# 1) Mean-field versus particle model under reduced stochasticity.
_test_mech = simulate_mechanistic_focal(SINGLE_CELL_MEAN_PCN, n_points=400)
_test_particle = simulate_single_cell_pcn(
    mean_pcn=SINGLE_CELL_MEAN_PCN,
    n_cells=3000,
    initial_cv=0.0,
    control_noise=0.0,
    dt_h=0.05,
    seed=123,
    substrate_time=_test_mech['time_h'],
    substrate_values=_test_mech['substrate'],
)
_m = float(_test_mech['pcn'][0, -1])
_p = float(np.mean(_test_particle['final_pcn']))
_gap = abs(_p - _m) / max(_m, 1e-12)
print('Mean-field endpoint PCN:', round(_m, 2))
print('Low-noise particle endpoint mean:', round(_p, 2))
print('Low-noise relative endpoint gap:', round(_gap, 4))

# Observed gap is ~0.2%; 5% leaves Monte Carlo headroom while still
# rejecting the implementation asymmetry an earlier version tolerated.
LOW_NOISE_CROSSMODEL_TOL = 0.05
assert _gap < LOW_NOISE_CROSSMODEL_TOL, (_m, _p, _gap)

# 2) Validate the ACTUAL plotted/shipped stochastic run against the 100-copy
# focal trajectory in the ACTUAL plotted equal-share 20-strain reference arena.
# This covers the configuration and substrate trace used by the figures.
_m_shipped = float(mech['pcn'][0, -1])
_p_shipped = float(np.mean(single['final_pcn']))
_shipped_gap = abs(_p_shipped - _m_shipped) / max(_m_shipped, 1e-12)
print('Plotted mean-field endpoint PCN:', round(_m_shipped, 2))
print('Plotted shipped-config particle endpoint mean:', round(_p_shipped, 2))
print('Plotted shipped-config relative gap:', round(_shipped_gap, 4))

SHIPPED_CROSSMODEL_TOL = 0.10
assert _shipped_gap < SHIPPED_CROSSMODEL_TOL, (_m_shipped, _p_shipped, _shipped_gap)
assert _p_shipped <= 1.02 * _m_shipped, (_m_shipped, _p_shipped)  # selection should not raise the mean materially

# 3) Endpoint heterogeneity must respond materially to initial CV.
_low = simulate_single_cell_pcn(
    n_cells=2500, initial_cv=0.05, control_noise=0.05, dt_h=0.10, seed=456,
    substrate_time=mech['time_h'], substrate_values=mech['substrate'])
_high = simulate_single_cell_pcn(
    n_cells=2500, initial_cv=0.60, control_noise=0.05, dt_h=0.10, seed=456,
    substrate_time=mech['time_h'], substrate_values=mech['substrate'])
_cv_low_initial = observed_cv(_low['final_pcn'])
_cv_high_initial = observed_cv(_high['final_pcn'])
print('Endpoint CV low/high initial:', round(_cv_low_initial,3), round(_cv_high_initial,3))
assert _cv_high_initial > 1.5 * _cv_low_initial

# 4) Endpoint heterogeneity must respond materially to control noise.
_low = simulate_single_cell_pcn(
    n_cells=2500, initial_cv=0.20, control_noise=0.0, dt_h=0.10, seed=789,
    substrate_time=mech['time_h'], substrate_values=mech['substrate'])
_high = simulate_single_cell_pcn(
    n_cells=2500, initial_cv=0.20, control_noise=0.30, dt_h=0.10, seed=789,
    substrate_time=mech['time_h'], substrate_values=mech['substrate'])
_cv_low_noise = observed_cv(_low['final_pcn'])
_cv_high_noise = observed_cv(_high['final_pcn'])
print('Endpoint CV low/high noise:', round(_cv_low_noise,3), round(_cv_high_noise,3))
assert _cv_high_noise > 1.5 * _cv_low_noise

# 5) Halving dt should not materially change endpoint CV.
_a = simulate_single_cell_pcn(
    n_cells=3000, initial_cv=0.30, control_noise=0.15, dt_h=0.10, seed=321,
    substrate_time=mech['time_h'], substrate_values=mech['substrate'])
_b = simulate_single_cell_pcn(
    n_cells=3000, initial_cv=0.30, control_noise=0.15, dt_h=0.05, seed=321,
    substrate_time=mech['time_h'], substrate_values=mech['substrate'])
_cv_dt_a = observed_cv(_a['final_pcn'])
_cv_dt_b = observed_cv(_b['final_pcn'])
_dt_rel_diff = abs(_cv_dt_a-_cv_dt_b) / max((_cv_dt_a+_cv_dt_b)/2, 1e-12)
print('Endpoint CV dt=.10/.05:', round(_cv_dt_a,3), round(_cv_dt_b,3), 'relative diff', round(_dt_rel_diff,3))
assert _dt_rel_diff < 0.20

## 6. Fixed-growth versus fixed-pool focal yield

Each candidate is evaluated in a separate counterfactual flask against the same declared synthetic pool, at the same focal inoculum fraction. Curves show **absolute focal plasmid yield relative to the same 100-copy anchor within each model**. Their ratio compares relative performance, not absolute prediction error or within-flask barcode fractions. The anchor, competitors, duration and initial fractions are modelling inputs; plotting resolution is not.

The loss curves in this section use the **core sensitivity range (0, 0.1%, 1% per division)**. The deliberately extreme 5–10% loss cases are reserved for the stress-test table in section 11.


In [ ]:
# @title Cross-check analytic vs shared model across loss rates
comparison_pcns = np.array(example_pcns, dtype=float)
rows = []

fig, ax = new_fig(w=9.2, h=5.8)
_loss_colors = seq_colors(len(LOSS_SWEEP_VALUES), lo=1, hi=6)
_curve_cache = []

for loss_value, color in zip(LOSS_SWEEP_VALUES, _loss_colors):
    static = log2_score(comparison_pcns, MATCHED_G0, loss_per_division=loss_value)
    static = 2 ** (static - log2_score(COMPARISON_ANCHOR_PCN, MATCHED_G0, loss_per_division=loss_value))

    se = focal_grid(comparison_pcns, loss_per_division=loss_value)[:,1]
    anchor = evaluate_focal(COMPARISON_ANCHOR_PCN, loss_per_division=loss_value)[1]
    shared = se / anchor
    ratio = static / shared
    _curve_cache.append((loss_value, color, ratio))

    for p, s1, s2, r in zip(comparison_pcns, static, shared, ratio):
        rows.append({
            'loss_per_division': loss_value,
            'PCN': int(p),
            'fixed_G0_normalized': s1,
            'shared_resource_normalized': s2,
            'fixed_over_shared': r,
        })

_ymin = min(np.min(r) for _,_,r in _curve_cache)
_ymax = max(np.max(r) for _,_,r in _curve_cache)
_pad = 0.08 * max(_ymax-_ymin, 0.1)
ax.set_ylim(_ymin-_pad, _ymax+_pad)

for loss_value, color, ratio in _curve_cache:
    ax.plot(comparison_pcns, ratio, marker='o', color=color, linewidth=2.1, markersize=4)
    direct_label(ax, comparison_pcns[-1], ratio[-1], f'loss={loss_value:.3g}', color, dx=9, weight='normal')

ax.axhline(1.0, color=theme('ink2'), linewidth=1.0)
ax.set_xlabel('Plasmid copy number per cell')
ax.set_ylabel('fixed-$G_0$ score ÷ shared-pool score  (>1 = shortcut optimistic)')
clean_axes(ax)
fig_header(
    fig,
    'Relative performance depends on the competition model',
    'Synthetic 19 × PCN 100 pool; focal fraction 1/20; core loss sweep only; each model anchored at PCN 100.',
)
fig.subplots_adjust(top=0.82, right=0.86)
savefig('15_fixed_vs_shared_bias_loss_sweep.png')
plt.show()

comparison = pd.DataFrame(rows)
display(comparison.round(3))
print(REFERENCE_POOL_LABEL, 'Focal fraction:', FOCAL_FRACTION)


## 7. Sensitivity to the published burden-fit uncertainty

Rouches et al. report a fitted burden coefficient of approximately

\[
0.065\% \pm 0.006\%
\]

per copy, with the errors calculated from variance estimates of the fitted coefficients.

The interval below propagates **that fit uncertainty only**. It is **not** a posterior interval for BL21(DE3), and it does not include uncertainty due to host strain, medium, plasmid length, cargo expression, or model misspecification.
This section applies only to the **analytic** optimum. A positive-truncated normal distribution for beta is an explicit propagation assumption. This is not an uncertainty interval for the shared-pool optimum.


In [ ]:
# @title Analytic-only burden coefficient propagation
from scipy.stats import truncnorm
rng=np.random.default_rng(42)
if BETA_SD_PUBLISHED<=0: raise ValueError('BETA_SD_PUBLISHED must be > 0.')
dist=truncnorm(-BETA/BETA_SD_PUBLISHED,np.inf,loc=BETA,scale=BETA_SD_PUBLISHED)
beta_samples=dist.rvs(size=100000,random_state=rng)
copt_samples=analytic_optimal_pcn(REFERENCE_G0,beta=beta_samples,loss_per_division=LOSS_PER_DIVISION)
exact_interval=analytic_optimal_pcn(REFERENCE_G0,beta=dist.ppf([.975,.025]),loss_per_division=LOSS_PER_DIVISION)
print('Assumption: normal beta conditional on beta > 0; analytic model only.')
if np.all(np.isfinite(copt_samples)):
    plt.figure(figsize=(8,5)); plt.hist(copt_samples,bins=80)
    plt.xlabel('Analytic interior optimum PCN'); plt.ylabel('Monte Carlo count')
    plt.title('Analytic model only: conditional-normal coefficient propagation')
    savefig('16_copt_fit_uncertainty.png'); plt.show()
    print('MC median / interval:',np.median(copt_samples),np.quantile(copt_samples,[.025,.975]))
    print('Exact transformed quantile interval:',exact_interval)
else:
    print('No analytic interior optimum for this loss setting; histogram omitted.')
beta_grid=np.linspace(max(1e-6,BETA-4*BETA_SD_PUBLISHED),BETA+4*BETA_SD_PUBLISHED,300)
cstar_grid=analytic_optimal_pcn(REFERENCE_G0,beta=beta_grid,loss_per_division=LOSS_PER_DIVISION)
if np.any(np.isfinite(cstar_grid)):
    plt.figure(figsize=(8,5)); plt.plot(beta_grid,cstar_grid); plt.axvline(BETA,linewidth=1)
    plt.xlabel('Burden coefficient beta'); plt.ylabel('Analytic interior optimum PCN')
    plt.title('Analytic optimum sensitivity; excludes model-transfer uncertainty')
    savefig('17_copt_vs_beta.png'); plt.show()


## 8. Exploratory plasmid-length scaling

Rouches et al. note that pUC19 is 2,686 bp and that their fitted per-copy cost is close to its fraction of the *E. coli* genome. Akeno et al. compared pUC-derived constructs of 2,334, 6,426 and 9,573 bp and found host growth correlated more strongly with total exogenous DNA amount than with molecule count alone.

That supports **PCN × DNA length as a biologically meaningful axis**, but it does not establish that the Rouches coefficient transfers linearly across hosts, cargos and plasmid sizes. The following cell therefore remains an exploratory sensitivity calculation.


In [ ]:
# @title Exploratory plasmid-length scaling
def exploratory_length_scaled_beta(plasmid_bp, beta=None, reference_bp=PUC19_BP):
    beta = _resolve_beta(beta)
    if plasmid_bp <= 0:
        raise ValueError("plasmid_bp must be > 0.")
    return beta * (plasmid_bp / reference_bp)

lengths = [2686, 3500, 5000, 8000]
rows = []

for bp in lengths:
    beta_eff = exploratory_length_scaled_beta(bp)
    rows.append({
        "Plasmid length (bp)": bp,
        "Exploratory beta": beta_eff,
        f"Predicted C* at G0={REFERENCE_G0:g}": analytic_optimal_pcn(
            REFERENCE_G0, beta=beta_eff, loss_per_division=LOSS_PER_DIVISION
        ),
    })

display(pd.DataFrame(rows).round(4))

## 9. Fixed-pool focal carrier versus plasmid yield

Each point represents a separate counterfactual candidate against the same synthetic competitors. Both axes are absolute focal endpoint quantities, displayed relative to the same 100-copy reference candidate. These are pooled-culture proxies; **the carrier axis is not the monoculture award measurement**. Dominance is conditional on this pool and burden law. A separate monoculture experiment/model is required to compare the two actual awards.


In [ ]:
# @title Fixed-pool Pareto comparison and model reconciliation
def pareto_mask_maximize(x,y):
    x=np.asarray(x,float); y=np.asarray(y,float)
    return np.array([not np.any((x>=a)&(y>=b)&((x>a)|(y>b))) for a,b in zip(x,y)])

pareto_pcns=np.arange(10,VALIDATED_PCN_MAX+1,10)
focal=focal_grid(pareto_pcns,loss_per_division=LOSS_PER_DIVISION)
carrier,absolute_plasmids=focal.T
anchor=np.array(evaluate_focal(COMPARISON_ANCHOR_PCN,loss_per_division=LOSS_PER_DIVISION))
carrier_norm=carrier/anchor[0]; plasmid_norm=absolute_plasmids/anchor[1]
is_front=pareto_mask_maximize(carrier,absolute_plasmids)
shared_optimum,shared_optimum_value=bounded_focal_optimum('absolute_plasmids',loss_per_division=LOSS_PER_DIVISION)

plt.figure(figsize=(9,6)); ax=plt.gca()
sc=ax.scatter(carrier_norm,plasmid_norm,c=pareto_pcns,cmap=PCN_CMAP,s=30)
idx=np.where(is_front)[0]; idx=idx[np.argsort(carrier_norm[idx])]
ax.plot(carrier_norm[idx],plasmid_norm[idx],color=DARK,label='Conditional Pareto front')

# Restore the headline and a few anchors directly on the figure.
_m=evaluate_focal_metrics(shared_optimum,loss_per_division=LOSS_PER_DIVISION)
_x=_m['carrier_cells']/anchor[0]; _y=_m['absolute_plasmids']/anchor[1]
direct_label(ax,_x,_y,f"C* ≈ {shared_optimum:.0f}",theme('ink'),dx=10)
for p in (50,100,400):
    if p in pareto_pcns:
        i=int(np.where(pareto_pcns==p)[0][0])
        direct_label(ax,carrier_norm[i],plasmid_norm[i],str(p),theme('ink2'),dx=6,weight='normal')

ax.set_xlabel('Focal carrier cells / carrier cells at PCN 100')
ax.set_ylabel('Absolute focal plasmid proxy / value at PCN 100')
editorial_title(
    'Carrier and absolute plasmid yields under one declared synthetic ecology',
    REFERENCE_POOL_LABEL+'; '+FOCAL_FRACTION_LABEL,
    ax=ax,
)
plt.colorbar(sc,ax=ax,label='Candidate PCN'); clean_axes(ax); ax.legend(); plt.subplots_adjust(top=.80)
savefig('18_pareto_shared_resource.png'); plt.show()

pareto_table=pd.DataFrame({
    'PCN':pareto_pcns,
    'carrier_relative_to_anchor':carrier_norm,
    'absolute_plasmids_relative_to_anchor':plasmid_norm,
    'pareto_optimal':is_front,
})
print(REFERENCE_POOL_LABEL,';',FOCAL_FRACTION_LABEL,'; hours',SHARED_HOURS)
print('Shared-model bounded optimum for absolute focal plasmid yield:',shared_optimum)
display(pd.DataFrame([
 {'model':'Analytic, configured G0','G0':REFERENCE_G0,'optimum':float(analytic_optimal_pcn(REFERENCE_G0)),'scope':'Interior formula; fit propagation in section 7'},
 {'model':'Analytic, matched log2(K/N0)','G0':MATCHED_G0,'optimum':float(analytic_optimal_pcn(MATCHED_G0)),'scope':'Interior formula; not a shared-model interval'},
 {'model':'Shared, declared synthetic reference ecology','G0':np.nan,'optimum':shared_optimum,'scope':'Absolute focal plasmid proxy; bounded 10..800; no uncertainty interval'},
]))
print('These optima are conditional; no fixed percentage gap is assumed.')


## 10. Ecological-context sensitivity and measurement mapping

The fixed-pool architecture removes a software artifact: a candidate's prediction no longer changes when the analyst changes the **plotting grid**. It does **not** remove real ecological dependence, nor should it.

The notebook therefore exposes five scenario levers explicitly:

1. **Growth opportunity** \(K/N_0\): how many population doublings are available before the batch approaches carrying capacity.
2. **Focal starting fraction**: the candidate's share of the initial inoculum.
3. **Synthetic competitor composition**: several deliberately artificial 19-competitor pools, never inferred from participant submissions.
4. **Culture duration**: checked separately to determine whether the shipped 12 h setting lies on the saturation plateau.
5. **Synthetic pool size**: explored separately from pool composition; non-20-team runs are explicitly warned and use equal inoculation for the actual number of lineages.

The section also reports two endpoint readouts side by side:

- **absolute focal plasmid proxy** = focal carrier cells × focal PCN;
- **within-pool barcode fraction** = focal plasmid proxy / total plasmid proxy.

The Grand Arena uses barcode counting after pooled plasmid extraction, but the supplied rules do not fully specify library-input normalization and read-count mapping. The notebook therefore does not silently promote either mathematical proxy to the official scoring observable.


In [ ]:
# @title Ecological-context sensitivity and measurement mapping
# Reference readout comparison: absolute focal plasmid yield versus within-pool barcode fraction.
_pcns=np.arange(10,VALIDATED_PCN_MAX+1,10,dtype=float)
_abs=focal_metric_grid(_pcns,'absolute_plasmids',loss_per_division=LOSS_PER_DIVISION)
_frac=focal_metric_grid(_pcns,'barcode_fraction',loss_per_division=LOSS_PER_DIVISION)
_abs_opt,_=bounded_focal_optimum('absolute_plasmids',loss_per_division=LOSS_PER_DIVISION)
_frac_opt,_=bounded_focal_optimum('barcode_fraction',loss_per_division=LOSS_PER_DIVISION)

measurement_mapping=pd.DataFrame([
    {'readout':'absolute focal plasmid proxy','optimum_PCN':_abs_opt,'interpretation':'depends on focal molecules produced'},
    {'readout':'within-pool barcode fraction','optimum_PCN':_frac_opt,'interpretation':'depends on focal molecules and denominator across competitors'},
])
display(measurement_mapping.round(2))

fig,ax=new_fig(w=9.2,h=5.8)
_abs_rel=_abs/np.max(_abs); _frac_rel=_frac/np.max(_frac)
ax.plot(_pcns,_abs_rel,color=theme('cat')[0],linewidth=2.3,label='absolute focal plasmid proxy')
ax.plot(_pcns,_frac_rel,color=theme('cat')[1],linewidth=2.3,label='within-pool barcode fraction')
ax.axvline(_abs_opt,color=theme('cat')[0],linewidth=1.0,alpha=.6)
ax.axvline(_frac_opt,color=theme('cat')[1],linewidth=1.0,alpha=.6)
ax.set_ylim(0.08,1.07)
direct_label(ax,_abs_opt,1.0,f'abs C*≈{_abs_opt:.0f}',theme('cat')[0],dx=8,dy=18,weight='normal')
direct_label(ax,_frac_opt,1.0,f'fraction C*≈{_frac_opt:.0f}',theme('cat')[1],dx=8,dy=-18,weight='normal')
ax.set_xlabel('Candidate PCN'); ax.set_ylabel('Readout / its own maximum')
clean_axes(ax); ax.legend()
fig_header(fig,'The measurement definition can move the optimum','Same declared synthetic pool; curves are normalized only for visual comparison.')
fig.subplots_adjust(top=.82,right=.86)
savefig('19_measurement_mapping.png'); plt.show()

# A. Growth opportunity sensitivity. Hold N0 fixed and vary K/N0 explicitly.
_growth_rows=[]
for _ratio in (1e2,1e3,1e4):
    _K=TOTAL_STARTING_CELLS*_ratio
    _o,_=bounded_focal_optimum('absolute_plasmids',carrying_capacity=_K,total_n0=TOTAL_STARTING_CELLS,loss_per_division=LOSS_PER_DIVISION)
    _growth_rows.append({'K_over_N0':_ratio,'log2_K_over_N0':np.log2(_ratio),'absolute_yield_optimum_PCN':_o})
growth_opportunity_sensitivity=pd.DataFrame(_growth_rows)
print('Growth-opportunity sensitivity (declared synthetic reference pool):')
display(growth_opportunity_sensitivity.round(2))

# B. Focal inoculum fraction sensitivity.
_fraction_rows=[]
for _ff in (0.025,0.05,0.10,0.20):
    _oa,_=bounded_focal_optimum('absolute_plasmids',focal_fraction=_ff,loss_per_division=LOSS_PER_DIVISION)
    _of,_=bounded_focal_optimum('barcode_fraction',focal_fraction=_ff,loss_per_division=LOSS_PER_DIVISION)
    _fraction_rows.append({'focal_fraction':_ff,'absolute_yield_optimum_PCN':_oa,'barcode_fraction_optimum_PCN':_of})
focal_fraction_sensitivity=pd.DataFrame(_fraction_rows)
print('Focal-fraction sensitivity:')
display(focal_fraction_sensitivity.round(2))

# C. Synthetic competitor composition sensitivity. These are illustrative pools only.
_pool_rows=[]
for _competitor_pcn in (50.,100.,200.,500.,700.):
    _pool=np.repeat(_competitor_pcn,19)
    _oa,_=bounded_focal_optimum('absolute_plasmids',pool=_pool,focal_fraction=FOCAL_FRACTION,loss_per_division=LOSS_PER_DIVISION)
    _of,_=bounded_focal_optimum('barcode_fraction',pool=_pool,focal_fraction=FOCAL_FRACTION,loss_per_division=LOSS_PER_DIVISION)
    _pool_rows.append({'synthetic_pool':f'19 × PCN {int(_competitor_pcn)}','absolute_yield_optimum_PCN':_oa,'barcode_fraction_optimum_PCN':_of})
pool_composition_sensitivity=pd.DataFrame(_pool_rows)
print('Synthetic-pool composition sensitivity (not participant-derived):')
display(pool_composition_sensitivity.round(2))

# D. Duration plateau: when the batch saturates, extending time no longer changes this model's optimum.
_duration_rows=[]
for _hours in (4.,6.,8.,10.,12.,16.,24.,36.):
    _o,_=bounded_focal_optimum('absolute_plasmids',hours=_hours,loss_per_division=LOSS_PER_DIVISION)
    _m=evaluate_focal_metrics(_o,hours=_hours,loss_per_division=LOSS_PER_DIVISION)
    _duration_rows.append({'hours':_hours,'absolute_yield_optimum_PCN':_o,'final_occupancy_K':_m['occupancy']})
duration_sensitivity=pd.DataFrame(_duration_rows)
print('Duration sensitivity:')
display(duration_sensitivity.round({'absolute_yield_optimum_PCN':1,'final_occupancy_K':3}))

_reference_occ=float(duration_sensitivity.loc[duration_sensitivity['hours']==SHARED_HOURS,'final_occupancy_K'].iloc[0]) if SHARED_HOURS in set(duration_sensitivity['hours']) else evaluate_focal_metrics(_abs_opt,hours=SHARED_HOURS,loss_per_division=LOSS_PER_DIVISION)['occupancy']
if _reference_occ>0.99:
    print(f'At SHARED_HOURS={SHARED_HOURS:g}, final occupancy is {_reference_occ:.3f}: the reference scenario lies on the saturation plateau.')
else:
    print(f'At SHARED_HOURS={SHARED_HOURS:g}, final occupancy is {_reference_occ:.3f}: duration remains an active model assumption.')


# E. Synthetic pool-size sensitivity. Up to 20 finalists is the public Arena framing;
# these runs ask how the model behaves if fewer lineages actually enter the flask.
_poolsize_rows=[]
for _n_total in (10,15,20):
    _pool=np.repeat(100.0,_n_total-1)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',RuntimeWarning)
        _oa,_=bounded_puc_focal_optimum(pool_pcns=_pool,focal_fraction=None,metric='absolute_plasmids')
        _of,_=bounded_puc_focal_optimum(pool_pcns=_pool,focal_fraction=None,metric='barcode_fraction')
    _poolsize_rows.append({
        'total_lineages':_n_total,
        'equal_start_fraction':1.0/_n_total,
        'absolute_yield_optimum_PCN':_oa,
        'barcode_fraction_optimum_PCN':_of,
    })
pool_size_sensitivity=pd.DataFrame(_poolsize_rows)
print('Synthetic pool-size sensitivity (equal inoculation; all competitors PCN 100):')
display(pool_size_sensitivity.round(3))

print('Interpretation: plotting-grid invariance is a correctness requirement; ecological-context invariance is not.')


## 11. All-pUC 20-strain exploration

This section adopts the simplifying assumption: **all 20 lineages use pUC/ColE1-family replication**. It remains synthetic and does not represent participant submissions.

Two complementary scenarios are shown:

1. **Engineered pUC continuum:** 20 equal-share lineages span the Rouches-style 20–800 PCN range. All plasmids are otherwise pUC19-sized with zero added cargo burden. This isolates copy-number ecology.
2. **Focal design sensitivity:** one focal pUC lineage is varied in PCN, plasmid length and a generic saturating cargo burden while the other 19 remain the declared PCN-100 pUC19 reference pool.

The extended growth model uses

\[
b_{DNA}=\beta C(L/2686)
\]

and

\[
b_{cargo}=E_{max}C/(K_E+C).
\]

Only the pUC19 per-copy coefficient is directly fitted in Rouches et al. Length transfer and cargo saturation are explicit scenario assumptions. Akeno et al. support total DNA amount as a better burden axis than copy count alone, and Wang et al. show that low- and high-copy ColE1 plasmids in BL21 alter growth and central metabolism in ways not captured by a single resource scalar.

The loss sweep deliberately distinguishes a **core sensitivity range** (up to 1% per division) from 5–10% stress tests. The review also highlighted the unresolved contrast between the no-cargo Rouches fit (~0.065% per copy) and the sfGFP-bearing example (~15% growth reduction near 50 copies). A one-point reconciliation diagnostic asks: it asks what generic cargo parameter would be required for the extended model to reproduce that approximate sfGFP point, without treating the result as a universal fit. PCN-dependent loss is shown only as a sensitivity mechanism. Finally, the single-cell CV ladder includes CV = 1 because direct pUC measurements show dispersion of approximately that order.


In [ ]:
# @title All-pUC 20-strain and parameter exploration
# A. One fully synthetic 20-member engineered-pUC continuum, equal 5% cell fractions.
_s20_pcns=np.asarray(SYNTHETIC_20_PCN,float)
_s20=simulate_puc_fixed_batch(
    _s20_pcns,
    plasmid_bp=np.repeat(PUC19_BP,ARENA_TEAM_COUNT),
    cargo_burden_max=np.zeros(ARENA_TEAM_COUNT),
    initial_fractions=np.repeat(EQUAL_START_FRACTION,ARENA_TEAM_COUNT),
)
assert np.allclose(_s20['design_cell_fraction'][:,0],EQUAL_START_FRACTION)

_s20_table=pd.DataFrame({
    'starting_PCN':_s20_pcns,
    'starting_cell_fraction':_s20['design_cell_fraction'][:,0],
    'starting_barcode_fraction':_s20['barcode_fraction'][:,0],
    'endpoint_lineage_fraction':_s20['design_cell_fraction'][:,-1],
    'endpoint_barcode_fraction':_s20['barcode_fraction'][:,-1],
})
print('Synthetic 20-member pUC continuum; all lineages start at cell fraction 0.05.')
display(_s20_table.round(4))

fig,ax=new_fig(w=9.2,h=5.8)
ax.plot(_s20_table['starting_PCN'],_s20_table['endpoint_lineage_fraction'],marker='o',linewidth=1.8,label='endpoint lineage fraction')
ax.plot(_s20_table['starting_PCN'],_s20_table['endpoint_barcode_fraction'],marker='o',linewidth=1.8,label='endpoint barcode fraction')
ax.axhline(EQUAL_START_FRACTION,color=theme('muted'),linewidth=1,linestyle='--')
ax.set_xscale('log')
ax.set_xticks([20,50,100,200,400,800],[20,50,100,200,400,800])
ax.set_xlabel('Starting PCN (synthetic pUC continuum)'); ax.set_ylabel('Endpoint fraction')
clean_axes(ax); ax.legend()
fig_header(fig,'Twenty equal-share pUC lineages separate in cell and molecule space','Synthetic continuum only; all are pUC19-sized and use zero additional cargo burden.')
fig.subplots_adjust(top=.82); savefig('20_synthetic_20_puc_continuum.png'); plt.show()

# B. Length × cargo burden: resolution-independent focal optimum against the 19×PCN100 pool.
_length_grid=[PUC19_BP,3500.,5000.,8000.]
_cargo_grid=[0.00,0.05,0.15,0.30]
_lc_rows=[]
for _bp in _length_grid:
    for _cargo in _cargo_grid:
        _o,_v=bounded_puc_focal_optimum(candidate_bp=_bp,candidate_cargo=_cargo,metric='absolute_plasmids')
        _of,_=bounded_puc_focal_optimum(candidate_bp=_bp,candidate_cargo=_cargo,metric='barcode_fraction')
        _lc_rows.append({'plasmid_bp':_bp,'cargo_burden_max':_cargo,'absolute_yield_optimum_PCN':_o,'barcode_fraction_optimum_PCN':_of})
length_cargo_sensitivity=pd.DataFrame(_lc_rows)
print('Length × generic cargo sensitivity; cargo_burden_max is a scenario parameter, not a fitted coefficient.')
display(length_cargo_sensitivity.round(1))

_pivot=length_cargo_sensitivity.pivot(index='plasmid_bp',columns='cargo_burden_max',values='absolute_yield_optimum_PCN')
fig,ax=new_fig(w=8.6,h=5.8)
im=ax.imshow(_pivot.values,aspect='auto',cmap=PCN_CMAP)
ax.set_xticks(np.arange(len(_pivot.columns)),[f'{x:.0%}' for x in _pivot.columns])
ax.set_yticks(np.arange(len(_pivot.index)),[f'{int(x):,}' for x in _pivot.index])
for iy in range(_pivot.shape[0]):
    for ix in range(_pivot.shape[1]):
        ax.text(ix,iy,f'{_pivot.values[iy,ix]:.0f}',ha='center',va='center',color=theme('ink'))
ax.set_xlabel('Maximum generic cargo growth penalty'); ax.set_ylabel('Focal plasmid length (bp)')
plt.colorbar(im,ax=ax,label='Absolute-yield optimum PCN')
fig_header(fig,'Longer or more expression-burdensome pUC constructs favor lower PCN','Scenario transfer of pUC19 DNA burden; numbers are not BL21(DE3) fits.')
fig.subplots_adjust(top=.82); savefig('21_length_cargo_optimum.png'); plt.show()

# B2. One-point Rouches no-cargo vs sfGFP consistency diagnostic.
# Approximate literature-reading target used only as a diagnostic, not a re-fit:
# total growth penalty ~0.15 at PCN~50 for an sfGFP-bearing construct.
_ROUCHES_SFGFP_PCN=50.0
_ROUCHES_SFGFP_TOTAL_PENALTY=0.15
_r_dna=BETA*_ROUCHES_SFGFP_PCN*(PUC19_BP/PUC19_BP)
_r_required_cargo=max(0.0,_ROUCHES_SFGFP_TOTAL_PENALTY-_r_dna)
_r_factor=_ROUCHES_SFGFP_PCN/(PUC_CARGO_HALF_SAT_PCN+_ROUCHES_SFGFP_PCN)
_r_implied_emax=_r_required_cargo/_r_factor
_r_reconstructed=1.0-float(puc_growth_factor(_ROUCHES_SFGFP_PCN,PUC19_BP,_r_implied_emax))
rouches_cargo_reconciliation=pd.DataFrame([{
    'approx_PCN':_ROUCHES_SFGFP_PCN,
    'approx_total_growth_penalty':_ROUCHES_SFGFP_TOTAL_PENALTY,
    'DNA_penalty_from_0.065pct_per_copy':_r_dna,
    'cargo_penalty_required_at_that_PCN':_r_required_cargo,
    'implied_generic_cargo_Emax_at_K=100':_r_implied_emax,
    'model_reconstructed_total_penalty':_r_reconstructed,
}])
print('Rouches no-cargo/sfGFP one-point consistency diagnostic (approximate, not a universal cargo fit):')
display(rouches_cargo_reconciliation.round(4))

# C. Loss: literature-scale core plus explicit stress tests.
_loss_values=[0.0,1e-5,1e-4,1e-3,1e-2,0.05,0.10]
_loss_rows=[]
for _loss in _loss_values:
    _o,_=bounded_puc_focal_optimum(candidate_bp=PUC19_BP,candidate_cargo=0.0,loss_per_division=_loss,loss_pcn_exponent=0.0)
    _loss_rows.append({'loss_per_division_at_all_PCN':_loss,'absolute_yield_optimum_PCN':_o,'class':'stress test' if _loss>=0.05 else 'core sweep'})
loss_scale_sensitivity=pd.DataFrame(_loss_rows)
print('Constant-loss sensitivity:')
display(loss_scale_sensitivity.round({'loss_per_division_at_all_PCN':5,'absolute_yield_optimum_PCN':1}))

_pcnloss_rows=[]
for _exp in (0.0,0.5,1.0):
    _o,_=bounded_puc_focal_optimum(candidate_bp=PUC19_BP,candidate_cargo=0.0,loss_per_division=0.01,loss_reference_pcn=100,loss_pcn_exponent=_exp)
    _pcnloss_rows.append({'loss_at_PCN100':0.01,'PCN_loss_exponent':_exp,'absolute_yield_optimum_PCN':_o})
pcn_loss_sensitivity=pd.DataFrame(_pcnloss_rows)
print('Phenomenological PCN-dependent-loss sensitivity:')
display(pcn_loss_sensitivity.round(2))

# D. Single-cell pUC heterogeneity ladder using the same reference substrate trajectory.
_cv_rows=[]
for _cv in (0.30,0.60,1.00):
    _sc=simulate_single_cell_pcn(n_cells=3000,initial_cv=_cv,control_noise=SINGLE_CELL_CONTROL_NOISE,seed=100+int(_cv*100),substrate_time=mech['time_h'],substrate_values=mech['substrate'])
    _end=np.asarray(_sc['final_pcn'],float)
    _cv_rows.append({'initial_CV_target':_cv,'endpoint_mean_PCN':np.mean(_end),'endpoint_CV':observed_cv(_end),'endpoint_plasmid_free_fraction':np.mean(_end==0)})
heterogeneity_sensitivity=pd.DataFrame(_cv_rows)
print('Single-cell pUC heterogeneity sensitivity:')
display(heterogeneity_sensitivity.round(3))

print('Interpretation: these are structured sensitivity scenarios, not probability distributions over the actual 20 teams.')


## 12. Suggested report wording

> We evaluated copy-number strategies using the fitted first-order relationship reported for near-isogenic ColE1/pUC19 variants, approximately \(0.065\%\pm0.006\%\) host growth cost per plasmid copy. Under the simplifying assumptions of fixed copy number, negligible segregational loss and reference growth opportunity \(G_0\), expected endpoint plasmid abundance scales as \(Q\propto C\,2^{G_0(1-\beta C)}\). The preferred copy number therefore depends strongly on the available growth opportunity. We cross-check this analytic model against declared 20-strain shared-resource scenarios in which equal-cell inoculation means each lineage begins at 0.05. Numerical optima are scenario results, not BL21(DE3)-specific predictions.

### Recommended interpretation hierarchy

1. Use the **analytic model** for intuition and closed-form break-even calculations.
2. Use the **fixed-pool shared-resource model** as a declared illustrative ecology, not as “the Battle Royale optimum” until inoculum and growth conditions are experimentally specified.
3. For equal inoculation of 20 strains, set **cell fraction** to 0.05. Do not force barcode fraction to 0.05: it is molecule-weighted and depends on PCN from the start.
4. Report **absolute focal plasmid yield** and **within-pool barcode fraction** separately. The sequencing/library workflow determines which is closer to the eventual scoring observable.
5. Treat the **pUC length/cargo extension** and **dynamic-copy model** as hypothesis generators. Only the pUC19 no-cargo per-copy coefficient is directly anchored to the Rouches fit; length transfer, cargo saturation and PCN-dependent loss are scenario mechanisms. The sfGFP one-point diagnostic only shows that the two Rouches readings can be represented within this decomposition; it is not a universal cargo fit.
6. Use the all-pUC 20-strain continuum only to expose qualitative competition structure. It is not an inferred field of actual participant designs.
7. Use the **Pareto plot** for dominance conditional on the declared synthetic pool; its carrier axis is a pooled-culture quantity and is **not** the Monoculture Award measurement.
8. Keep participant-derived strategy data outside this shareable notebook.

### Ecological-context wording

> Fixing the competitor pool removes a plotting artifact, not ecological dependence. In the shared-resource model, the numerical optimum can shift when \(K/N_0\), focal inoculum fraction, synthetic competitor composition, construct length, expression burden or loss changes. These sweeps are therefore shown explicitly. Once the current simple batch reaches carrying capacity, extending time has little effect; this robustness should not be extrapolated to models with stationary-phase death, ongoing plasmid loss or DNA persistence.

### Mechanistic-extension wording

> We additionally explored a pUC/ColE1-focused resource model in which cells consume a shared normalized substrate and plasmid copy number responds dynamically to growth state. A focal mechanistic trajectory is evaluated against nineteen unchanged reference competitors at a 5% starting cell fraction. Growth-rate dependence of ColE1 copy-number control is supported by prior work, but the feedback strength and relaxation time are scenario parameters rather than fitted BL21(DE3) constants.

### Heterogeneity wording

> Population-average PCN can conceal broad single-cell variation. Direct single-cell measurements of pUC reported a standard deviation on the order of the mean copy number. We therefore sweep initial PCN CV up to 1.0, but treat the variance parameters as sensitivity values until the competition constructs are measured directly.

The section 7 uncertainty interval belongs only to the analytic burden-fit calculation. Shared-model and pUC-extended optima are conditional scenario outputs. Do not transfer the analytic interval to them or quote any one PCN optimum as universal.


In [ ]:
# @title Final QA audit
assert expect_raises(lambda:relative_growth(0),ValueError)
assert expect_raises(lambda:validate_mechanistic_raw_state(np.array([[100.]]),np.array([-0.5]),DYNAMIC_PCN_CAP),RuntimeError)
_c0=float(analytic_optimal_pcn(REFERENCE_G0,loss_per_division=0.0)); _c001=float(analytic_optimal_pcn(REFERENCE_G0,loss_per_division=0.001)); _c01=float(analytic_optimal_pcn(REFERENCE_G0,loss_per_division=0.01)); assert _c0<_c001<_c01
assert hasattr(np,'trapezoid')
print('QA checks passed.')
print('Core loss-aware analytic optima:',{'loss=0':round(_c0,1),'loss=.001':round(_c001,1),'loss=.01':round(_c01,1)})

# Cross-model tolerances used by the stochastic validation are intentionally stated.
assert _gap < LOW_NOISE_CROSSMODEL_TOL, _gap
assert _shipped_gap < SHIPPED_CROSSMODEL_TOL, _shipped_gap
print('Cross-model tolerances:', {'low_noise': LOW_NOISE_CROSSMODEL_TOL, 'shipped': SHIPPED_CROSSMODEL_TOL})

# Plotting-grid/order invariance: identical focal candidates must have identical absolute yields.
_a=focal_grid([100,200,700]); _b=focal_grid([700,50,100,200])
assert np.allclose(_a,_b[[2,3,0]],rtol=1e-10)

# Inoculum validation.
assert expect_raises(lambda:simulate_shared_batch([100,200],initial_fractions=[.2,.2]),ValueError)
assert expect_raises(lambda:simulate_shared_batch([100,200],initial_fractions=[1.1,-.1]),ValueError)

# Invalid analytic optimum regions preserve shape and become NaN rather than negative/infinite PCNs.
assert np.all(np.isnan(analytic_optimal_pcn(10,beta=np.array([.0006,.0007]),loss_per_division=.7)))
assert analytic_optimal_pcn(10,beta=np.array([.0006,.0007]),loss_per_division=.7).shape==(2,)

# Measurement readouts are intentionally distinct APIs.
_m100=evaluate_focal_metrics(100)
assert np.isclose(_m100['absolute_plasmids'],_m100['carrier_cells']*100)
assert 0 < _m100['barcode_fraction'] < 1

# The shipped 12 h default may be treated as duration-insensitive only if the reference batch saturates.
_ref_occ=evaluate_focal_metrics(shared_optimum,hours=SHARED_HOURS,loss_per_division=LOSS_PER_DIVISION)['occupancy']
assert _ref_occ>0.99, f'flask not saturated at SHARED_HOURS={SHARED_HOURS}; optimum remains duration-dependent (occupancy={_ref_occ:.3f})'

# Growth opportunity is deliberately an active ecological parameter: this is a sensitivity sanity check, not invariance.
_g=growth_opportunity_sensitivity.sort_values('K_over_N0')
assert np.all(np.diff(_g['absolute_yield_optimum_PCN'].to_numpy()) < 0), 'Expected optimum to decrease as available growth opportunity increases in this scenario.'

print('Fixed-pool invariance, readout separation, saturation guard, and ecological-sensitivity checks passed.')


# Equal-share Arena checks.
assert ARENA_TEAM_COUNT==20 and np.isclose(EQUAL_START_FRACTION,0.05)
assert len(REFERENCE_POOL_PCNS)==19
assert all(np.isclose(run['initial_fractions'][0],0.05) for run in mech_runs.values())
assert np.allclose(_s20['design_cell_fraction'][:,0],0.05)

# pUC extension must exactly recover the original Rouches growth law at 2,686 bp, zero cargo.
_test_p=np.array([50.,100.,300.])
assert np.allclose(puc_growth_factor(_test_p,PUC19_BP,0.0),relative_growth(_test_p))

# Fault-sensitive length-law checks away from the inert ratio=1 reference point.
_bp,_p=3500.0,100.0
_expected=1.0-BETA*_p*(_bp/2686.0)
assert np.isclose(float(puc_growth_factor(_p,_bp,0.0)),_expected,rtol=0,atol=1e-12)
_a=1.0-float(puc_growth_factor(_p,2.0*2686.0,0.0))
_b=1.0-float(puc_growth_factor(_p,1.0*2686.0,0.0))
assert np.isclose(_a,2.0*_b,rtol=1e-12),(_a,_b)

# Length and cargo burdens must not improve growth at fixed PCN.
assert puc_growth_factor(100,5000,0.0) < puc_growth_factor(100,PUC19_BP,0.0)
assert puc_growth_factor(100,PUC19_BP,0.15) < puc_growth_factor(100,PUC19_BP,0.0)

# The first-order pUC extension must refuse states whose combined burden predicts <=0 growth.
assert expect_raises(lambda:puc_burden_components(800.,8000.,0.30),ValueError)
puc_burden_components(100.,8000.,0.30)

# Non-Arena pool sizes are allowed as explicit synthetic sensitivity runs, and
# unspecified focal fraction means equal inoculation for the actual pool size.
with warnings.catch_warnings():
    warnings.simplefilter('ignore',RuntimeWarning)
    _custom=simulate_puc_focal_fixed(100,pool_pcns=np.repeat(100.,18),focal_fraction=None)
assert np.isclose(_custom['design_cell_fraction'][0,0],1/19)

# PCN-dependent loss exponent zero must recover the constant-loss model.
assert np.allclose(pcn_dependent_loss_per_division(np.array([50.,100.,500.]),base_loss=0.01,exponent=0.0),0.01)
print('Equal-share, pUC-extension, burden-domain, and pool-size QA checks passed.')


## 13. Key literature behind the model

References are held centrally in [`evidence/references.bib`](../../evidence/references.bib)
and cited by key, so this notebook does not maintain a second bibliography.

**Copy-number burden, tunability, and cargo diagnostic**
- [@rouches2022tunable] — the primary quantitative anchor. The no-cargo
  per-copy coefficient is stated with its parameterisation under MODEL
  STATUS above. The approximate sfGFP reading (~15% growth reduction near
  ~50 copies) is used only for a one-point decomposition diagnostic, never
  as a universal per-copy coefficient.

**Total exogenous DNA / plasmid-length burden**
- [@akeno2014reducedgenome] — pUC-derived test plasmids of 2,334, 6,426 and
  9,573 bp; growth related more strongly to total exogenous DNA amount than
  to copy number alone in their analysis.

**Growth-rate-dependent ColE1 copy-number control**
- [@klumpp2011growthrate]

**BL21 metabolism under low- versus high-copy ColE1 plasmids**
- [@wang2006cole1] — approximately 50-copy and ~410-copy ColE1-derived
  plasmids, with changes in growth, glucose and oxygen use, acetate
  secretion and central metabolism.

**Single-cell pUC copy-number distributions**
- [@shao2021singlecell] — pUC mean copy number about 61 with standard
  deviation about 60, motivating the CV ≈ 1 stress test.
- [@jahn2016copynumber]

**Batch physiology / resource depletion in LB**
- [@sezonov2007lb]

**Figure design**
- [@wong2010datafigures], [@midway2020visualization], [@jambor2025checklist]
